In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import gc
import math
import copy
import json
import time
import random
import joblib
import psutil
import threading
import subprocess
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

!pip -q install thop
from thop import profile

In [ ]:
# paths

DATA_PATH = "/content/drive/MyDrive/PES/energyDataset/energydata_complete.csv"

PROJECT_ROOT    = "/content/drive/MyDrive/PES/project"
MODELS_ROOT     = os.path.join(PROJECT_ROOT, "models")
REPORTS_ROOT    = os.path.join(PROJECT_ROOT, "reports")
CHECKPOINT_ROOT = os.path.join(PROJECT_ROOT, "checkpoints")
ARTIFACTS_ROOT  = os.path.join(PROJECT_ROOT, "artifacts")
RESOURCES_ROOT  = os.path.join(PROJECT_ROOT, "resources")
LOGS_ROOT       = os.path.join(PROJECT_ROOT, "logs")

BASE_MODEL_DIR = os.path.join(MODELS_ROOT, "baseModel")
BASE_LOGS_DIR = os.path.join(LOGS_ROOT, "base") # Added BASE_LOGS_DIR here

VAN_DIR        = os.path.join(MODELS_ROOT, "vanStudents")
VAN_ADV_DIR    = os.path.join(MODELS_ROOT, "vanStudents_adv")

NOV_DIR        = os.path.join(MODELS_ROOT, "novStudents")
NOV_ADV_DIR    = os.path.join(MODELS_ROOT, "novStudents_adv")

FGSM_ADV_DIR = os.path.join(VAN_ADV_DIR, "fgsm")
BIM_ADV_DIR  = os.path.join(VAN_ADV_DIR, "bim")
PGD_ADV_DIR  = os.path.join(VAN_ADV_DIR, "pgd")

FGSM_LOGS_DIR = os.path.join(LOGS_ROOT, "fgsm")
BIM_LOGS_DIR  = os.path.join(LOGS_ROOT, "bim")
PGD_LOGS_DIR  = os.path.join(LOGS_ROOT, "pgd")

FGSM_CKPT_DIR = os.path.join(CHECKPOINT_ROOT, "fgsm")
BIM_CKPT_DIR  = os.path.join(CHECKPOINT_ROOT, "bim")
PGD_CKPT_DIR  = os.path.join(CHECKPOINT_ROOT, "pgd")
BASE_CKPT_DIR = os.path.join(CHECKPOINT_ROOT, "base")

# definitions for results directories
FGSM_RESULTS_DIR = os.path.join(REPORTS_ROOT, "fgsm_results")
BIM_RESULTS_DIR  = os.path.join(REPORTS_ROOT, "bim_results")
PGD_RESULTS_DIR  = os.path.join(REPORTS_ROOT, "pgd_results")

for d in [
    PROJECT_ROOT, MODELS_ROOT, REPORTS_ROOT, CHECKPOINT_ROOT,
    ARTIFACTS_ROOT, RESOURCES_ROOT, LOGS_ROOT,
    BASE_MODEL_DIR, VAN_DIR, VAN_ADV_DIR, NOV_DIR, NOV_ADV_DIR,
    FGSM_ADV_DIR, BIM_ADV_DIR, PGD_ADV_DIR,
    FGSM_LOGS_DIR, BIM_LOGS_DIR, PGD_LOGS_DIR, BASE_LOGS_DIR,
    FGSM_CKPT_DIR, BIM_CKPT_DIR, PGD_CKPT_DIR, BASE_CKPT_DIR,
    FGSM_RESULTS_DIR, BIM_RESULTS_DIR, PGD_RESULTS_DIR # Added new result directories
]:
    os.makedirs(d, exist_ok=True)

print("Dataset path:", DATA_PATH)
print("Project root :", PROJECT_ROOT)


FGSM_RUNS_CSV   = os.path.join(FGSM_RESULTS_DIR, "fgsm_morphence_runs.csv")
FGSM_STATS_CSV  = os.path.join(FGSM_RESULTS_DIR, "fgsm_morphence_stats.csv")
FGSM_WEIGHT_DIVERSITY_CSV = os.path.join(FGSM_RESULTS_DIR, "fgsm_weight_diversity.csv")
FGSM_PRED_DIVERSITY_CSV   = os.path.join(FGSM_RESULTS_DIR, "fgsm_prediction_diversity.csv")
FGSM_XFER_MATRIX_CSV      = os.path.join(FGSM_RESULTS_DIR, "fgsm_transfer_matrix.csv")

BIM_RUNS_CSV    = os.path.join(BIM_RESULTS_DIR, "bim_morphence_runs.csv")
BIM_STATS_CSV   = os.path.join(BIM_RESULTS_DIR, "bim_morphence_stats.csv")
BIM_WEIGHT_DIVERSITY_CSV  = os.path.join(BIM_RESULTS_DIR, "bim_weight_diversity.csv")
BIM_PRED_DIVERSITY_CSV    = os.path.join(BIM_RESULTS_DIR, "bim_prediction_diversity.csv")
BIM_XFER_MATRIX_CSV       = os.path.join(BIM_RESULTS_DIR, "bim_transfer_matrix.csv")

PGD_RUNS_CSV    = os.path.join(PGD_RESULTS_DIR, "pgd_morphence_runs.csv")
PGD_STATS_CSV   = os.path.join(PGD_RESULTS_DIR, "pgd_morphence_stats.csv")
PGD_WEIGHT_DIVERSITY_CSV  = os.path.join(PGD_RESULTS_DIR, "pgd_weight_diversity.csv")
PGD_PRED_DIVERSITY_CSV    = os.path.join(PGD_RESULTS_DIR, "pgd_prediction_diversity.csv")
PGD_XFER_MATRIX_CSV       = os.path.join(PGD_RESULTS_DIR, "pgd_transfer_matrix.csv")

BASE_TRAIN_HISTORY_CSV = os.path.join(BASE_LOGS_DIR, "jena_base_train_history.csv")

Dataset path: /content/drive/MyDrive/PES/energyDataset/energydata_complete.csv
Project root : /content/drive/MyDrive/PES/project


In [ ]:
# configs

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print("Device:", device)

# windowing
FREQ = "10min"
STEPS_PER_HOUR = 6
HOURS_PER_DAY  = 24

HIST_DAYS = 30
HZN_DAYS  = 7

HIST_STEPS = HIST_DAYS * HOURS_PER_DAY * STEPS_PER_HOUR   # 4320 data points
HZN_STEPS  = HZN_DAYS  * HOURS_PER_DAY * STEPS_PER_HOUR   # 1008
SLIDE      = 1

TRAIN_FRAC = 0.70
VAL_FRAC   = 0.10

# data loader
BATCH = 16
NUM_WORKERS = 2
PIN_MEMORY = True

# model
PATCH    = 12
D_MODEL  = 256
N_HEAD   = 4
N_LAYERS = 3
DIM_FF   = 512
DROPOUT  = 0.1
D_OUT    = 1

# base training
LR_BASE     = 5e-4
EPOCHS_BASE = 15

# vanilla students
NUM_STUDENTS = 4
NOISE_STD    = 1e-3

# student adversarial training
LR_STUDENT   = 1e-3
FGSM_EPOCHS  = 5
BIM_EPOCHS   = 5
PGD_EPOCHS   = 5

FGSM_TRAIN_EPS_LIST = [0.1, 0.2, 0.3, 0.5]
BIM_TRAIN_EPS_LIST  = [0.1, 0.2, 0.3, 0.5]
PGD_TRAIN_EPS_LIST  = [0.1, 0.2, 0.3, 0.5]

FGSM_ALPHA = 0.02
BIM_ITERS  = 10
PGD_ITERS  = 10

Q_LOSS_Q = 0.5

# pipeline switches
RUN_PREPROCESS       = True
RUN_BASE_TRAIN       = True
RUN_BASE_TEST        = True
RUN_COMPLEXITY       = True
RUN_STUDENT_GEN      = True
RUN_STUDENT_CLEAN    = True
RUN_FGSM_ADV_TRAIN   = True
RUN_BIM_ADV_TRAIN    = True
RUN_PGD_ADV_TRAIN    = True

# evaluation switches
RUN_FGSM_EVAL = True
RUN_BIM_EVAL  = True
RUN_PGD_EVAL  = True

Device: cuda


In [ ]:
# artifacts

# preprocessing
X_TRAIN_PATH = os.path.join(ARTIFACTS_ROOT, "X_train.npy")
Y_TRAIN_PATH = os.path.join(ARTIFACTS_ROOT, "Y_train.npy")
X_VAL_PATH   = os.path.join(ARTIFACTS_ROOT, "X_val.npy")
Y_VAL_PATH   = os.path.join(ARTIFACTS_ROOT, "Y_val.npy")
X_TEST_PATH  = os.path.join(ARTIFACTS_ROOT, "X_test.npy")
Y_TEST_PATH  = os.path.join(ARTIFACTS_ROOT, "Y_test.npy")

META_TRAIN_PATH = os.path.join(ARTIFACTS_ROOT, "meta_train.csv")
META_VAL_PATH   = os.path.join(ARTIFACTS_ROOT, "meta_val.csv")
META_TEST_PATH  = os.path.join(ARTIFACTS_ROOT, "meta_test.csv")

X_SCALER_PATH = os.path.join(ARTIFACTS_ROOT, "x_scaler.joblib")
Y_SCALER_PATH = os.path.join(ARTIFACTS_ROOT, "y_scaler.joblib")

CONFIG_JSON_PATH = os.path.join(ARTIFACTS_ROOT, "run_config.json")

# base outputs
BASE_BEST_PATH      = os.path.join(BASE_MODEL_DIR, "appliance_base_best.pth")
BASE_RESUME_CKPT    = os.path.join(BASE_CKPT_DIR, "base_resume_checkpoint.pt")
BASE_TRAIN_LOG_CSV  = os.path.join(LOGS_ROOT, "base_train_log.csv")

# clean evaluation
VANILLA_CLEAN_CSV   = os.path.join(REPORTS_ROOT, "vanilla_students_clean_eval.csv")
VANILLA_COMPARE_CSV = os.path.join(REPORTS_ROOT, "vanilla_students_clean_compare.csv")

# complexity report
COMPLEXITY_JSON     = os.path.join(REPORTS_ROOT, "base_model_complexity.json")

# global resource report
PIPELINE_RESOURCE_SUMMARY_CSV = os.path.join(RESOURCES_ROOT, "pipeline_resource_summary.csv")

In [ ]:
# resource monitor

class ResourceMonitor:
    def __init__(self, sample_interval=2.0):
        self.sample_interval = sample_interval
        self._stop_event = threading.Event()
        self._thread = None
        self.records = []
        self.max_cpu_ram_mb = 0.0
        self.max_gpu_mem_mb = 0.0

    def _query_gpu(self):
        try:
            cmd = [
                "nvidia-smi",
                "--query-gpu=utilization.gpu,memory.used,memory.total,power.draw",
                "--format=csv,noheader,nounits"
            ]
            out = subprocess.check_output(cmd).decode("utf-8").strip()
            vals = [v.strip() for v in out.split(",")]

            gpu_util = float(vals[0]) if vals[0] not in ["N/A", "[Not Supported]"] else None
            mem_used = float(vals[1]) if vals[1] not in ["N/A", "[Not Supported]"] else None
            mem_total = float(vals[2]) if vals[2] not in ["N/A", "[Not Supported]"] else None
            power_w = float(vals[3]) if vals[3] not in ["N/A", "[Not Supported]"] else None

            return {
                "gpu_util_percent": gpu_util,
                "gpu_mem_used_mb": mem_used,
                "gpu_mem_total_mb": mem_total,
                "gpu_power_w": power_w,
            }
        except Exception:
            return {
                "gpu_util_percent": None,
                "gpu_mem_used_mb": None,
                "gpu_mem_total_mb": None,
                "gpu_power_w": None,
            }

    def _sample_once(self):
        ts = time.time()

        proc = psutil.Process(os.getpid())
        cpu_ram_mb = proc.memory_info().rss / (1024 ** 2)
        self.max_cpu_ram_mb = max(self.max_cpu_ram_mb, cpu_ram_mb)

        torch_gpu_mem_mb = None
        if torch.cuda.is_available():
            try:
                torch_gpu_mem_mb = torch.cuda.memory_allocated() / (1024 ** 2)
                self.max_gpu_mem_mb = max(self.max_gpu_mem_mb, torch_gpu_mem_mb)
            except Exception:
                pass

        gpu_stats = self._query_gpu()
        if gpu_stats["gpu_mem_used_mb"] is not None:
            self.max_gpu_mem_mb = max(self.max_gpu_mem_mb, gpu_stats["gpu_mem_used_mb"])

        self.records.append({
            "timestamp": ts,
            "cpu_ram_mb": cpu_ram_mb,
            "torch_gpu_mem_mb": torch_gpu_mem_mb,
            **gpu_stats
        })

    def _run(self):
        while not self._stop_event.is_set():
            self._sample_once()
            time.sleep(self.sample_interval)

    def start(self):
        self.records = []
        self.max_cpu_ram_mb = 0.0
        self.max_gpu_mem_mb = 0.0
        self._stop_event.clear()
        self._thread = threading.Thread(target=self._run, daemon=True)
        self._thread.start()

    def stop(self):
        self._stop_event.set()
        if self._thread is not None:
            self._thread.join(timeout=2)

    def to_dataframe(self):
        return pd.DataFrame(self.records)

    def summary(self):
        if len(self.records) == 0:
            return {}

        dfm = pd.DataFrame(self.records).sort_values("timestamp").reset_index(drop=True)

        avg_gpu_util = None
        gpu_active_pct = None
        if dfm["gpu_util_percent"].notna().any():
            avg_gpu_util = float(dfm["gpu_util_percent"].dropna().mean())
            gpu_active_pct = float((dfm["gpu_util_percent"] > 0).mean() * 100.0)

        est_gpu_energy_j = None
        if dfm["gpu_power_w"].notna().sum() >= 2:
            t = dfm["timestamp"].values
            p = dfm["gpu_power_w"].astype(float).values
            mask = ~np.isnan(p)
            if mask.sum() >= 2:
                est_gpu_energy_j = float(np.trapezoid(p[mask], t[mask]))

        return {
            "n_samples": int(len(dfm)),
            "sample_interval_sec": self.sample_interval,
            "max_cpu_ram_mb": float(self.max_cpu_ram_mb),
            "max_gpu_mem_mb": float(self.max_gpu_mem_mb),
            "avg_gpu_util_percent": avg_gpu_util,
            "gpu_active_percent_of_samples": gpu_active_pct,
            "est_gpu_energy_j": est_gpu_energy_j,
        }

In [ ]:
# resource tracked stage wrapper

pipeline_resource_rows = []

def run_stage_with_monitor(stage_name, fn, *args, sample_interval=2.0, **kwargs):
    print(f"\n{'='*20} STAGE: {stage_name} {'='*20}")
    stage_dir = os.path.join(RESOURCES_ROOT, stage_name)
    os.makedirs(stage_dir, exist_ok=True)

    trace_csv = os.path.join(stage_dir, f"{stage_name}_resource_trace.csv")
    summary_csv = os.path.join(stage_dir, f"{stage_name}_resource_summary.csv")

    monitor = ResourceMonitor(sample_interval=sample_interval)
    start_time = time.time()
    monitor.start()

    result = None
    try:
        result = fn(*args, **kwargs)
    finally:
        monitor.stop()

    elapsed_min = (time.time() - start_time) / 60.0

    trace_df = monitor.to_dataframe()
    summary = monitor.summary()
    summary["stage_name"] = stage_name
    summary["stage_elapsed_min"] = elapsed_min

    trace_df.to_csv(trace_csv, index=False)
    pd.DataFrame([summary]).to_csv(summary_csv, index=False)

    pipeline_resource_rows.append(summary)
    pd.DataFrame(pipeline_resource_rows).to_csv(PIPELINE_RESOURCE_SUMMARY_CSV, index=False)

    print("Stage resource summary:")
    print(summary)

    return result

In [ ]:
# checkpoint helpers

def save_resume_checkpoint(path, model, optimizer, epoch, history, best_metric=None, extra=None):
    ckpt = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict() if optimizer is not None else None,
        "epoch": epoch,
        "history": history,
        "best_metric": best_metric,
        "extra": extra if extra is not None else {}
    }
    torch.save(ckpt, path)

def load_resume_checkpoint(path, model, optimizer=None, map_location="cpu"):
    ckpt = torch.load(path, map_location=map_location)
    model.load_state_dict(ckpt["model_state_dict"])
    if optimizer is not None and ckpt.get("optimizer_state_dict") is not None:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    return ckpt

def latest_epoch_checkpoint(ckpt_dir, prefix):
    files = [
        os.path.join(ckpt_dir, f)
        for f in os.listdir(ckpt_dir)
        if f.startswith(prefix) and f.endswith(".pt")
    ]
    if len(files) == 0:
        return None
    files = sorted(files)
    return files[-1]

In [ ]:
# preprocessing helpers

target_col = "Appliances"
time_col   = "date"

drop_cols = ["rv1", "rv2"]

indoor_temp_cols = [f"T{i}" for i in range(1, 10)]
indoor_rh_cols   = [f"RH_{i}" for i in range(1, 10)]
weather_cols     = ["T_out", "Press_mm_hg", "RH_out", "Windspeed", "Visibility", "Tdewpoint"]
behavior_cols    = ["lights"]

feature_cols = behavior_cols + indoor_temp_cols + indoor_rh_cols + weather_cols
D_IN = len(feature_cols)

assert HIST_STEPS % PATCH == 0, "HIST_STEPS must be divisible by PATCH"
assert HZN_STEPS % PATCH == 0, "HZN_STEPS must be divisible by PATCH"

print("D_IN :", D_IN)
print("D_OUT:", D_OUT)

D_IN : 25
D_OUT: 1


In [ ]:
# full preprocessing stage

def preprocess_and_save():
    df = pd.read_csv(DATA_PATH)
    print("Raw shape:", df.shape)

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.sort_values("date").reset_index(drop=True)

    print("Date range:", df["date"].min(), "->", df["date"].max())
    print("Missing values total:", int(df.isna().sum().sum()))
    print("Duplicate timestamps:", int(df["date"].duplicated().sum()))

    missing_expected = [c for c in feature_cols + [target_col, time_col] if c not in df.columns]
    assert len(missing_expected) == 0, f"Missing expected columns: {missing_expected}"

    df = df[[time_col, target_col] + feature_cols].copy()

    dt_diff = df[time_col].diff().dropna()
    expected_freq = pd.Timedelta(minutes=10)

    print("Unique time gaps:")
    print(dt_diff.value_counts().sort_index())

    assert (dt_diff == expected_freq).all(), "Time index is not perfectly 10 minute regular."
    assert df.isna().sum().sum() == 0, "There are still missing values."

    n_total = len(df)
    split_idx = int(TRAIN_FRAC * n_total)
    df_train_part = df.iloc[:split_idx].copy()

    x_scaler = MinMaxScaler()
    y_scaler = MinMaxScaler()

    x_scaler.fit(df_train_part[feature_cols].values)
    y_scaler.fit(df_train_part[[target_col]].values)

    X_full_scaled = x_scaler.transform(df[feature_cols].values).astype(np.float32)
    Y_full_scaled = y_scaler.transform(df[[target_col]].values).astype(np.float32)

    print("Scaled X shape:", X_full_scaled.shape)
    print("Scaled Y shape:", Y_full_scaled.shape)

    X_list, Y_list, meta = [], [], []
    T = len(df)

    for t in range(HIST_STEPS, T - HZN_STEPS, SLIDE):
        x_win = X_full_scaled[t - HIST_STEPS:t, :]
        y_win = Y_full_scaled[t:t + HZN_STEPS, :]

        X_list.append(x_win)
        Y_list.append(y_win)

        meta.append((
            df.iloc[t - HIST_STEPS][time_col],
            df.iloc[t - 1][time_col],
            df.iloc[t][time_col],
            df.iloc[t + HZN_STEPS - 1][time_col]
        ))

    X = np.stack(X_list).astype(np.float32)
    Y = np.stack(Y_list).astype(np.float32)
    meta_df = pd.DataFrame(meta, columns=["x_start", "x_end", "y_start", "y_end"])

    print("Windowed X shape:", X.shape)
    print("Windowed Y shape:", Y.shape)
    print("Number of windows:", len(X))
    print(meta_df.head())

    N = len(X)
    n_train = int(TRAIN_FRAC * N)
    n_val   = int(VAL_FRAC * N)
    n_test  = N - n_train - n_val

    idx_train = slice(0, n_train)
    idx_val   = slice(n_train, n_train + n_val)
    idx_test  = slice(n_train + n_val, N)

    X_train, Y_train = X[idx_train], Y[idx_train]
    X_val,   Y_val   = X[idx_val],   Y[idx_val]
    X_test,  Y_test  = X[idx_test],  Y[idx_test]

    meta_train = meta_df.iloc[idx_train].reset_index(drop=True)
    meta_val   = meta_df.iloc[idx_val].reset_index(drop=True)
    meta_test  = meta_df.iloc[idx_test].reset_index(drop=True)

    print("Train:", X_train.shape, Y_train.shape)
    print("Val  :", X_val.shape, Y_val.shape)
    print("Test :", X_test.shape, Y_test.shape)

    np.save(X_TRAIN_PATH, X_train)
    np.save(Y_TRAIN_PATH, Y_train)
    np.save(X_VAL_PATH, X_val)
    np.save(Y_VAL_PATH, Y_val)
    np.save(X_TEST_PATH, X_test)
    np.save(Y_TEST_PATH, Y_test)

    meta_train.to_csv(META_TRAIN_PATH, index=False)
    meta_val.to_csv(META_VAL_PATH, index=False)
    meta_test.to_csv(META_TEST_PATH, index=False)

    joblib.dump(x_scaler, X_SCALER_PATH)
    joblib.dump(y_scaler, Y_SCALER_PATH)

    config_payload = {
        "feature_cols": feature_cols,
        "target_col": target_col,
        "time_col": time_col,
        "HIST_DAYS": HIST_DAYS,
        "HZN_DAYS": HZN_DAYS,
        "HIST_STEPS": HIST_STEPS,
        "HZN_STEPS": HZN_STEPS,
        "PATCH": PATCH,
        "D_IN": D_IN,
        "D_OUT": D_OUT,
        "D_MODEL": D_MODEL,
        "N_HEAD": N_HEAD,
        "N_LAYERS": N_LAYERS,
        "DIM_FF": DIM_FF,
        "DROPOUT": DROPOUT,
        "BATCH": BATCH,
        "TRAIN_FRAC": TRAIN_FRAC,
        "VAL_FRAC": VAL_FRAC,
        "SLIDE": SLIDE,
        "SEED": SEED,
    }
    with open(CONFIG_JSON_PATH, "w") as f:
        json.dump(config_payload, f, indent=2)

    print("Saved preprocessing artifacts to:", ARTIFACTS_ROOT)

def load_preprocessed():
    X_train = np.load(X_TRAIN_PATH)
    Y_train = np.load(Y_TRAIN_PATH)
    X_val   = np.load(X_VAL_PATH)
    Y_val   = np.load(Y_VAL_PATH)
    X_test  = np.load(X_TEST_PATH)
    Y_test  = np.load(Y_TEST_PATH)

    meta_train = pd.read_csv(META_TRAIN_PATH)
    meta_val   = pd.read_csv(META_VAL_PATH)
    meta_test  = pd.read_csv(META_TEST_PATH)

    x_scaler = joblib.load(X_SCALER_PATH)
    y_scaler = joblib.load(Y_SCALER_PATH)

    print("Loaded preprocessing artifacts from disk.")
    print("Train:", X_train.shape, Y_train.shape)
    print("Val  :", X_val.shape, Y_val.shape)
    print("Test :", X_test.shape, Y_test.shape)

    return X_train, Y_train, X_val, Y_val, X_test, Y_test, meta_train, meta_val, meta_test, x_scaler, y_scaler

preproc_files_exist = all([
    os.path.exists(X_TRAIN_PATH), os.path.exists(Y_TRAIN_PATH),
    os.path.exists(X_VAL_PATH),   os.path.exists(Y_VAL_PATH),
    os.path.exists(X_TEST_PATH),  os.path.exists(Y_TEST_PATH),
    os.path.exists(META_TRAIN_PATH), os.path.exists(META_VAL_PATH), os.path.exists(META_TEST_PATH),
    os.path.exists(X_SCALER_PATH), os.path.exists(Y_SCALER_PATH),
])

if RUN_PREPROCESS:
    if preproc_files_exist:
        print("Preprocessing artifacts already exist. Loading them.")
    else:
        run_stage_with_monitor("preprocess", preprocess_and_save)

X_train, Y_train, X_val, Y_val, X_test, Y_test, meta_train, meta_val, meta_test, x_scaler, y_scaler = load_preprocessed()

Preprocessing artifacts already exist. Loading them.
Loaded preprocessing artifacts from disk.
Train: (10084, 4320, 25) (10084, 1008, 1)
Val  : (1440, 4320, 25) (1440, 1008, 1)
Test : (2883, 4320, 25) (2883, 1008, 1)


In [ ]:
# dataset + data loaders

class ApplianceSeqDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.from_numpy(X)
        self.Y = torch.from_numpy(Y)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

train_dl = DataLoader(
    ApplianceSeqDataset(X_train, Y_train),
    batch_size=BATCH,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

val_dl = DataLoader(
    ApplianceSeqDataset(X_val, Y_val),
    batch_size=BATCH,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

test_dl = DataLoader(
    ApplianceSeqDataset(X_test, Y_test),
    batch_size=BATCH,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

xb, yb = next(iter(train_dl))
print("One batch X:", xb.shape)
print("One batch Y:", yb.shape)

One batch X: torch.Size([16, 4320, 25])
One batch Y: torch.Size([16, 1008, 1])


In [ ]:
# patching helpers

def patchify_input(x, patch):
    if patch == 1:
        return x
    B, T, D = x.shape
    assert T % patch == 0, f"Input length {T} must be divisible by patch {patch}"
    x = x.reshape(B, T // patch, patch * D)
    return x

def unpatchify_output(y_patch, patch, T_out, d_out=1):
    if patch == 1:
        return y_patch[:, :T_out, :]
    B, Tp, Dp = y_patch.shape
    assert Dp == d_out * patch, f"Expected last dim {d_out * patch}, got {Dp}"
    y = y_patch.reshape(B, Tp * patch, d_out)
    return y[:, :T_out, :]

In [ ]:
# positional encoding + transformer

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=6000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) *
            (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class ApplianceSeq2SeqTransformer(nn.Module):
    def __init__(
        self,
        d_in,
        d_out=1,
        d_model=256,
        nhead=4,
        num_layers=3,
        dim_ff=512,
        dropout=0.1,
        hist_steps=4320,
        hzn_steps=1008,
        patch=12
    ):
        super().__init__()

        assert hist_steps % patch == 0, "hist_steps must be divisible by patch"
        assert hzn_steps % patch == 0, "hzn_steps must be divisible by patch"

        self.d_in = d_in
        self.d_out = d_out
        self.d_model = d_model
        self.hist_steps = hist_steps
        self.hzn_steps = hzn_steps
        self.patch = patch

        self.hist_tokens = hist_steps // patch
        self.hzn_tokens  = hzn_steps // patch

        self.in_proj = nn.Linear(d_in * patch, d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.enc_pos = PositionalEncoding(d_model, max_len=self.hist_tokens + 16)

        dec_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )
        self.decoder = nn.TransformerDecoder(dec_layer, num_layers=num_layers)
        self.dec_pos = PositionalEncoding(d_model, max_len=self.hzn_tokens + 16)

        self.out_proj = nn.Linear(d_model, d_out * patch)

        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, x):
        B, T, D = x.shape
        assert T == self.hist_steps, f"Expected hist_steps={self.hist_steps}, got {T}"
        assert D == self.d_in, f"Expected d_in={self.d_in}, got {D}"

        x_patch = patchify_input(x, self.patch)
        z = self.in_proj(x_patch)
        z = self.enc_pos(z)
        mem = self.encoder(z)

        tgt = torch.zeros(B, self.hzn_tokens, self.d_model, device=x.device)
        tgt = self.dec_pos(tgt)

        dec_out = self.decoder(tgt, mem)
        y_patch = self.out_proj(dec_out)

        y = unpatchify_output(
            y_patch,
            patch=self.patch,
            T_out=self.hzn_steps,
            d_out=self.d_out
        )

        return y

In [ ]:
# model sanity check

model = ApplianceSeq2SeqTransformer(
    d_in=D_IN,
    d_out=D_OUT,
    d_model=D_MODEL,
    nhead=N_HEAD,
    num_layers=N_LAYERS,
    dim_ff=DIM_FF,
    dropout=DROPOUT,
    hist_steps=HIST_STEPS,
    hzn_steps=HZN_STEPS,
    patch=PATCH
).to(device)

xb, yb = next(iter(train_dl))
xb = xb.to(device)
yb = yb.to(device)

with torch.no_grad():
    pred = model(xb)

print("Input batch shape :", xb.shape)
print("Target batch shape:", yb.shape)
print("Pred batch shape  :", pred.shape)

/tmp/ipykernel_399/2283914661.py:61: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


Input batch shape : torch.Size([16, 4320, 25])
Target batch shape: torch.Size([16, 1008, 1])
Pred batch shape  : torch.Size([16, 1008, 1])


In [ ]:
# losses + metrics + eval

mse_loss = nn.MSELoss()

def rmse_only(pred, true):
    mse = mse_loss(pred, true)
    return torch.sqrt(mse + 1e-12)

def quantile_loss(y_true, y_pred, q=0.5):
    e = y_true - y_pred
    return torch.mean(torch.maximum(q * e, (q - 1) * e))

def seq_rmse_np(y_true_np, y_pred_np):
    return float(np.sqrt(mean_squared_error(y_true_np.ravel(), y_pred_np.ravel())))

@torch.no_grad()
def eval_one_epoch(model, loader):
    model.eval()
    ys, ps = [], []

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        pred = model(xb)
        ys.append(yb.cpu().numpy())
        ps.append(pred.cpu().numpy())

    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return seq_rmse_np(ys, ps)

In [ ]:
# base training

def build_model():
    return ApplianceSeq2SeqTransformer(
        d_in=D_IN,
        d_out=D_OUT,
        d_model=D_MODEL,
        nhead=N_HEAD,
        num_layers=N_LAYERS,
        dim_ff=DIM_FF,
        dropout=DROPOUT,
        hist_steps=HIST_STEPS,
        hzn_steps=HZN_STEPS,
        patch=PATCH
    )

def train_base_resumable(epochs=EPOCHS_BASE, lr=LR_BASE):
    model = build_model().to(device)
    opt = optim.AdamW(model.parameters(), lr=lr)

    start_epoch = 1
    best_val = float("inf")
    history = []

    if os.path.exists(BASE_RESUME_CKPT):
        print("Resuming base training from:", BASE_RESUME_CKPT)
        ckpt = load_resume_checkpoint(BASE_RESUME_CKPT, model, opt, map_location=device)
        start_epoch = ckpt["epoch"] + 1
        best_val = ckpt.get("best_metric", float("inf"))
        history = ckpt.get("history", [])
    elif os.path.exists(BASE_BEST_PATH):
        print("Best base model exists. Loading it as initialization:", BASE_BEST_PATH)
        model.load_state_dict(torch.load(BASE_BEST_PATH, map_location=device))

    for ep in range(start_epoch, epochs + 1):
        model.train()
        train_loss_sum = 0.0
        train_count = 0
        ep_start = time.time()

        for xb, yb in train_dl:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            pred = model(xb)
            loss = rmse_only(pred, yb)

            opt.zero_grad()
            loss.backward()
            opt.step()

            bs = xb.size(0)
            train_loss_sum += loss.item() * bs
            train_count += bs

        train_rmse = train_loss_sum / max(1, train_count)
        val_rmse = eval_one_epoch(model, val_dl)

        ep_mins = (time.time() - ep_start) / 60.0

        row = {
            "epoch": ep,
            "train_rmse": train_rmse,
            "val_rmse": val_rmse,
            "epoch_time_min": ep_mins,
        }
        history.append(row)
        pd.DataFrame(history).to_csv(BASE_TRAIN_LOG_CSV, index=False)

        if val_rmse < best_val:
            best_val = val_rmse
            torch.save(model.state_dict(), BASE_BEST_PATH)
            print(f"[Base] Epoch {ep:02d} | train={train_rmse:.5f} | val={val_rmse:.5f} | saved best")

        save_resume_checkpoint(
            BASE_RESUME_CKPT,
            model=model,
            optimizer=opt,
            epoch=ep,
            history=history,
            best_metric=best_val,
            extra={"lr": lr}
        )

        print(f"[Base] Epoch {ep:02d} | train={train_rmse:.5f} | val={val_rmse:.5f} | best={best_val:.5f}")

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    return history

if RUN_BASE_TRAIN:
    run_stage_with_monitor("base_train", train_base_resumable, epochs=EPOCHS_BASE, lr=LR_BASE)


==================== STAGE: base_train ====================


/tmp/ipykernel_399/2283914661.py:61: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


Resuming base training from: /content/drive/MyDrive/PES/project/checkpoints/base/base_resume_checkpoint.pt
Stage resource summary:
{'n_samples': 5, 'sample_interval_sec': 2.0, 'max_cpu_ram_mb': 7191.8125, 'max_gpu_mem_mb': 642.0, 'avg_gpu_util_percent': 0.6, 'gpu_active_percent_of_samples': 20.0, 'est_gpu_energy_j': 543.9190665245056, 'stage_name': 'base_train', 'stage_elapsed_min': 0.16929935216903685}


In [ ]:
# reload best base model and evaluate

best_base = build_model().to(device)
best_base.load_state_dict(torch.load(BASE_BEST_PATH, map_location=device))
best_base.eval()

def evaluate_base():
    val_rmse_best = eval_one_epoch(best_base, val_dl)
    test_rmse_best = eval_one_epoch(best_base, test_dl)
    print(f"Best model val RMSE : {val_rmse_best:.5f}")
    print(f"Best model test RMSE: {test_rmse_best:.5f}")
    return {"val_rmse": val_rmse_best, "test_rmse": test_rmse_best}

if RUN_BASE_TEST:
    base_eval = run_stage_with_monitor("base_eval", evaluate_base)
else:
    base_eval = None

/tmp/ipykernel_399/2283914661.py:61: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)



==================== STAGE: base_eval ====================
Best model val RMSE : 0.08567
Best model test RMSE: 0.07979
Stage resource summary:
{'n_samples': 2, 'sample_interval_sec': 2.0, 'max_cpu_ram_mb': 7233.85546875, 'max_gpu_mem_mb': 688.0, 'avg_gpu_util_percent': 28.0, 'gpu_active_percent_of_samples': 50.0, 'est_gpu_energy_j': 168.24990563631056, 'stage_name': 'base_eval', 'stage_elapsed_min': 0.06778311729431152}


In [ ]:

# complexity report

def compute_model_complexity():
    dummy_x = torch.randn(1, HIST_STEPS, D_IN).to(device)
    macs, params = profile(best_base, inputs=(dummy_x,), verbose=False)
    flops_est = 2 * macs

    payload = {
        "macs": float(macs),
        "flops": float(flops_est),
        "params": float(params)
    }

    with open(COMPLEXITY_JSON, "w") as f:
        json.dump(payload, f, indent=2)

    print("Estimated MACs :", macs)
    print("Estimated FLOPs:", flops_est)
    print("Parameter count:", params)

    return payload

if RUN_COMPLEXITY:
    complexity_info = run_stage_with_monitor("complexity", compute_model_complexity)
else:
    complexity_info = None


==================== STAGE: complexity ====================
Estimated MACs : 380067840.0
Estimated FLOPs: 760135680.0
Parameter count: 1665292.0
Stage resource summary:
{'n_samples': 1, 'sample_interval_sec': 2.0, 'max_cpu_ram_mb': 7249.4296875, 'max_gpu_mem_mb': 688.0, 'avg_gpu_util_percent': 0.0, 'gpu_active_percent_of_samples': 0.0, 'est_gpu_energy_j': None, 'stage_name': 'complexity', 'stage_elapsed_min': 0.03380899826685588}


In [ ]:
# vanilla student generation

def perturb_model_gaussian(model, noise_std=1e-3):
    student = copy.deepcopy(model)
    with torch.no_grad():
        for _, param in student.named_parameters():
            if param.requires_grad:
                param.add_(torch.randn_like(param) * noise_std)
    return student

def generate_vanilla_students(num_students=NUM_STUDENTS, noise_std=NOISE_STD):
    base_for_students = build_model().to(device)
    base_for_students.load_state_dict(torch.load(BASE_BEST_PATH, map_location=device))
    base_for_students.eval()

    student_paths = []
    for i in range(num_students):
        save_path = os.path.join(VAN_DIR, f"vanilla_student_{i+1:02d}.pth")
        if os.path.exists(save_path):
            print("Student already exists, skipping:", save_path)
            student_paths.append(save_path)
            continue

        student = perturb_model_gaussian(base_for_students, noise_std=noise_std)
        torch.save(student.state_dict(), save_path)
        student_paths.append(save_path)
        print("Saved student:", save_path)

    return student_paths

if RUN_STUDENT_GEN:
    student_paths = run_stage_with_monitor("vanilla_student_generation", generate_vanilla_students)
else:
    student_paths = sorted([
        os.path.join(VAN_DIR, f)
        for f in os.listdir(VAN_DIR) if f.endswith(".pth")
    ])

print("Vanilla students:")
for p in student_paths:
    print(" ", p)


==================== STAGE: vanilla_student_generation ====================
Student already exists, skipping: /content/drive/MyDrive/PES/project/models/vanStudents/vanilla_student_01.pth
Student already exists, skipping: /content/drive/MyDrive/PES/project/models/vanStudents/vanilla_student_02.pth
Student already exists, skipping: /content/drive/MyDrive/PES/project/models/vanStudents/vanilla_student_03.pth
Student already exists, skipping: /content/drive/MyDrive/PES/project/models/vanStudents/vanilla_student_04.pth


/tmp/ipykernel_399/2283914661.py:61: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


Stage resource summary:
{'n_samples': 1, 'sample_interval_sec': 2.0, 'max_cpu_ram_mb': 7249.61328125, 'max_gpu_mem_mb': 688.0, 'avg_gpu_util_percent': 0.0, 'gpu_active_percent_of_samples': 0.0, 'est_gpu_energy_j': None, 'stage_name': 'vanilla_student_generation', 'stage_elapsed_min': 0.03380018870035807}
Vanilla students:
  /content/drive/MyDrive/PES/project/models/vanStudents/vanilla_student_01.pth
  /content/drive/MyDrive/PES/project/models/vanStudents/vanilla_student_02.pth
  /content/drive/MyDrive/PES/project/models/vanStudents/vanilla_student_03.pth
  /content/drive/MyDrive/PES/project/models/vanStudents/vanilla_student_04.pth


In [ ]:
# vanilla student clean evaluation

def evaluate_vanilla_students_clean():
    student_paths = sorted([
        os.path.join(VAN_DIR, f)
        for f in os.listdir(VAN_DIR)
        if f.endswith(".pth")
    ])

    clean_rows = []

    base_val_rmse = eval_one_epoch(best_base, val_dl)
    base_test_rmse = eval_one_epoch(best_base, test_dl)

    clean_rows.append({
        "model": "base",
        "val_rmse": base_val_rmse,
        "test_rmse": base_test_rmse
    })

    print(f"[base] val_rmse={base_val_rmse:.5f} | test_rmse={base_test_rmse:.5f}")

    for sp in student_paths:
        stu = build_model().to(device)
        stu.load_state_dict(torch.load(sp, map_location=device))
        stu.eval()

        name = os.path.basename(sp).replace(".pth", "")
        val_rmse = eval_one_epoch(stu, val_dl)
        test_rmse = eval_one_epoch(stu, test_dl)

        clean_rows.append({
            "model": name,
            "val_rmse": val_rmse,
            "test_rmse": test_rmse
        })

        print(f"[{name}] val_rmse={val_rmse:.5f} | test_rmse={test_rmse:.5f}")

    df_clean_eval = pd.DataFrame(clean_rows).sort_values("model").reset_index(drop=True)
    df_clean_eval.to_csv(VANILLA_CLEAN_CSV, index=False)

    df_compare = df_clean_eval.copy()
    base_row = df_compare[df_compare["model"] == "base"].iloc[0]
    base_val = base_row["val_rmse"]
    base_test = base_row["test_rmse"]

    df_compare["delta_val_vs_base"] = df_compare["val_rmse"] - base_val
    df_compare["delta_test_vs_base"] = df_compare["test_rmse"] - base_test
    df_compare.to_csv(VANILLA_COMPARE_CSV, index=False)

    print("\nClean evaluation table:")
    print(df_clean_eval)
    print("\nComparison vs base:")
    print(df_compare)

    return df_clean_eval

if RUN_STUDENT_CLEAN:
    vanilla_clean_df = run_stage_with_monitor("vanilla_clean_eval", evaluate_vanilla_students_clean)
else:
    vanilla_clean_df = pd.read_csv(VANILLA_CLEAN_CSV) if os.path.exists(VANILLA_CLEAN_CSV) else None


==================== STAGE: vanilla_clean_eval ====================
[base] val_rmse=0.08567 | test_rmse=0.07979


/tmp/ipykernel_399/2283914661.py:61: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


[vanilla_student_01] val_rmse=0.08646 | test_rmse=0.08063


/tmp/ipykernel_399/2283914661.py:61: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


[vanilla_student_02] val_rmse=0.08667 | test_rmse=0.08080


/tmp/ipykernel_399/2283914661.py:61: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


[vanilla_student_03] val_rmse=0.08621 | test_rmse=0.08053


/tmp/ipykernel_399/2283914661.py:61: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


[vanilla_student_04] val_rmse=0.08610 | test_rmse=0.08089

Clean evaluation table:
                model  val_rmse  test_rmse
0                base  0.085671   0.079786
1  vanilla_student_01  0.086462   0.080631
2  vanilla_student_02  0.086672   0.080796
3  vanilla_student_03  0.086212   0.080529
4  vanilla_student_04  0.086098   0.080886

Comparison vs base:
                model  val_rmse  test_rmse  delta_val_vs_base  \
0                base  0.085671   0.079786           0.000000   
1  vanilla_student_01  0.086462   0.080631           0.000791   
2  vanilla_student_02  0.086672   0.080796           0.001001   
3  vanilla_student_03  0.086212   0.080529           0.000541   
4  vanilla_student_04  0.086098   0.080886           0.000427   

   delta_test_vs_base  
0            0.000000  
1            0.000846  
2            0.001010  
3            0.000744  
4            0.001101  
Stage resource summary:
{'n_samples': 12, 'sample_interval_sec': 2.0, 'max_cpu_ram_mb': 7288.765625, 'm

In [ ]:
# attack definitions

def rfgsm_attack(model, X, Y, eps=0.1, alpha=0.02):
    was_training = model.training
    model.eval()

    X_adv = (X + torch.empty_like(X).uniform_(-eps, eps)).clamp(0.0, 1.0).detach()
    X_adv.requires_grad_(True)

    preds = model(X_adv)
    loss = mse_loss(preds, Y)
    loss.backward()

    X_adv = (X_adv + alpha * X_adv.grad.sign()).clamp(0.0, 1.0).detach()

    if was_training:
        model.train()

    return X_adv

def bim_attack(model, X, Y, eps=0.15, alpha=None, iters=10, random_start=False):
    if alpha is None:
        alpha = eps / max(1, iters)

    was_training = model.training
    model.eval()

    if random_start:
        X_adv = (X + torch.empty_like(X).uniform_(-eps, eps)).clamp(0.0, 1.0).detach()
    else:
        X_adv = X.clone().detach()

    X_adv.requires_grad_(True)

    for _ in range(iters):
        preds = model(X_adv)
        loss = mse_loss(preds, Y)

        if X_adv.grad is not None:
            X_adv.grad.zero_()

        loss.backward()
        X_adv = (X_adv + alpha * X_adv.grad.sign()).detach()

        delta = torch.clamp(X_adv - X, -eps, eps)
        X_adv = (X + delta).clamp(0.0, 1.0).detach()
        X_adv.requires_grad_(True)

    if was_training:
        model.train()

    return X_adv.detach()

def pgd_attack(model, X, Y, eps=0.15, alpha=None, iters=10, random_start=True):
    if alpha is None:
        alpha = eps / max(1, iters)

    was_training = model.training
    model.eval()

    if random_start:
        X_adv = (X + torch.empty_like(X).uniform_(-eps, eps)).clamp(0.0, 1.0).detach()
    else:
        X_adv = X.clone().detach()

    X_adv.requires_grad_(True)

    for _ in range(iters):
        preds = model(X_adv)
        loss = mse_loss(preds, Y)

        if X_adv.grad is not None:
            X_adv.grad.zero_()

        loss.backward()
        X_adv = (X_adv + alpha * X_adv.grad.sign()).detach()

        delta = torch.clamp(X_adv - X, -eps, eps)
        X_adv = (X + delta).clamp(0.0, 1.0).detach()
        X_adv.requires_grad_(True)

    if was_training:
        model.train()

    return X_adv.detach()

In [ ]:
# student adversarial training

def train_student_adv_resumable(
    student_path,
    attack_name,
    attack_fn,
    attack_out_dir,
    attack_logs_dir,
    attack_ckpt_dir,
    out_suffix,
    epochs,
    lr,
    eps_list,
    q=0.5,
    attack_kwargs_fn=None
):
    os.makedirs(attack_out_dir, exist_ok=True)
    os.makedirs(attack_logs_dir, exist_ok=True)
    os.makedirs(attack_ckpt_dir, exist_ok=True)

    student_stub = os.path.basename(student_path).replace(".pth", "")
    final_out = os.path.join(attack_out_dir, f"{student_stub}{out_suffix}.pth")
    hist_path = os.path.join(attack_logs_dir, f"{student_stub}{out_suffix}_train_history.csv")
    resume_path = os.path.join(attack_ckpt_dir, f"{student_stub}{out_suffix}_resume.pt")

    model = build_model().to(device)
    opt = optim.AdamW(model.parameters(), lr=lr)

    history = []
    start_epoch = 1

    if os.path.exists(resume_path):
        print(f"[{attack_name}] Resuming from:", resume_path)
        ckpt = load_resume_checkpoint(resume_path, model, opt, map_location=device)
        history = ckpt.get("history", [])
        start_epoch = ckpt["epoch"] + 1
    elif os.path.exists(final_out):
        print(f"[{attack_name}] Final model already exists, loading for continued training:", final_out)
        model.load_state_dict(torch.load(final_out, map_location=device), strict=False)
    else:
        model.load_state_dict(torch.load(student_path, map_location=device), strict=False)

    for ep in range(start_epoch, epochs + 1):
        model.train()
        tot_clean = 0.0
        tot_adv_mean = 0.0
        tot_by_eps = {eps: 0.0 for eps in eps_list}
        cnt = 0

        ep_start = time.time()

        for xb, yb in train_dl:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            pc = model(xb)
            Lc = rmse_only(pc, yb) + quantile_loss(yb, pc, q)

            adv_losses = []
            adv_losses_per_eps = {}

            for eps in eps_list:
                atk_kwargs = attack_kwargs_fn(eps) if attack_kwargs_fn is not None else {"eps": eps}
                xa = attack_fn(model, xb, yb, **atk_kwargs)
                model.train()
                pa = model(xa)
                La = rmse_only(pa, yb) + quantile_loss(yb, pa, q)

                adv_losses.append(La)
                adv_losses_per_eps[eps] = La

            if len(adv_losses) > 0:
                La_mean = torch.stack(adv_losses).mean()
            else:
                La_mean = 0.0 * Lc

            L = 0.25 * Lc + 0.75 * La_mean

            opt.zero_grad()
            L.backward()
            opt.step()

            bs = xb.size(0)
            cnt += bs
            tot_clean += Lc.item() * bs
            tot_adv_mean += La_mean.item() * bs

            for eps in eps_list:
                tot_by_eps[eps] += adv_losses_per_eps[eps].item() * bs

        ep_mins = (time.time() - ep_start) / 60.0
        clean_loss_epoch = tot_clean / max(1, cnt)
        adv_mean_epoch   = tot_adv_mean / max(1, cnt)

        row = {
            "epoch": ep,
            "clean_loss": clean_loss_epoch,
            "adv_loss_mean": adv_mean_epoch,
            "epoch_minutes": ep_mins,
        }
        for eps in eps_list:
            row[f"adv_loss_eps_{eps:.2f}"] = tot_by_eps[eps] / max(1, cnt)

        history.append(row)
        pd.DataFrame(history).to_csv(hist_path, index=False)

        torch.save(model.state_dict(), final_out)

        save_resume_checkpoint(
            resume_path,
            model=model,
            optimizer=opt,
            epoch=ep,
            history=history,
            best_metric=None,
            extra={
                "attack_name": attack_name,
                "student_path": student_path,
                "lr": lr
            }
        )

        print(
            f"[{attack_name} train {student_stub}] "
            f"epoch {ep}/{epochs} | clean={clean_loss_epoch:.4f} | adv_mean={adv_mean_epoch:.4f}"
        )

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    print(f"Saved {attack_name} adv student:", final_out)
    print(f"Saved {attack_name} train history:", hist_path)

    return final_out


In [ ]:
# attack-specific wrappers

def train_all_fgsm_students():
    student_paths = sorted([
        os.path.join(VAN_DIR, f) for f in os.listdir(VAN_DIR) if f.endswith(".pth")
    ])
    outs = []

    for sp in student_paths:
        out = train_student_adv_resumable(
            student_path=sp,
            attack_name="FGSM",
            attack_fn=rfgsm_attack,
            attack_out_dir=FGSM_ADV_DIR,
            attack_logs_dir=FGSM_LOGS_DIR,
            attack_ckpt_dir=FGSM_CKPT_DIR,
            out_suffix="_fgsm_adv",
            epochs=FGSM_EPOCHS,
            lr=LR_STUDENT,
            eps_list=FGSM_TRAIN_EPS_LIST,
            q=Q_LOSS_Q,
            attack_kwargs_fn=lambda eps: {"eps": eps, "alpha": FGSM_ALPHA}
        )
        outs.append(out)

    return outs

def train_all_bim_students():
    student_paths = sorted([
        os.path.join(VAN_DIR, f) for f in os.listdir(VAN_DIR) if f.endswith(".pth")
    ])
    outs = []

    for sp in student_paths:
        out = train_student_adv_resumable(
            student_path=sp,
            attack_name="BIM",
            attack_fn=bim_attack,
            attack_out_dir=BIM_ADV_DIR,
            attack_logs_dir=BIM_LOGS_DIR,
            attack_ckpt_dir=BIM_CKPT_DIR,
            out_suffix="_bim_adv",
            epochs=BIM_EPOCHS,
            lr=LR_STUDENT,
            eps_list=BIM_TRAIN_EPS_LIST,
            q=Q_LOSS_Q,
            attack_kwargs_fn=lambda eps: {
                "eps": eps,
                "alpha": eps / max(1, BIM_ITERS),
                "iters": BIM_ITERS,
                "random_start": False
            }
        )
        outs.append(out)

    return outs

def train_all_pgd_students():
    student_paths = sorted([
        os.path.join(VAN_DIR, f) for f in os.listdir(VAN_DIR) if f.endswith(".pth")
    ])
    outs = []

    for sp in student_paths:
        out = train_student_adv_resumable(
            student_path=sp,
            attack_name="PGD",
            attack_fn=pgd_attack,
            attack_out_dir=PGD_ADV_DIR,
            attack_logs_dir=PGD_LOGS_DIR,
            attack_ckpt_dir=PGD_CKPT_DIR,
            out_suffix="_pgd_adv",
            epochs=PGD_EPOCHS,
            lr=LR_STUDENT,
            eps_list=PGD_TRAIN_EPS_LIST,
            q=Q_LOSS_Q,
            attack_kwargs_fn=lambda eps: {
                "eps": eps,
                "alpha": eps / max(1, PGD_ITERS),
                "iters": PGD_ITERS,
                "random_start": True
            }
        )
        outs.append(out)

    return outs

In [ ]:
# adversarial training for vanilla students

if RUN_FGSM_ADV_TRAIN:
    fgsm_adv_paths = run_stage_with_monitor("fgsm_adv_train", train_all_fgsm_students)
else:
    fgsm_adv_paths = sorted([
        os.path.join(FGSM_ADV_DIR, f) for f in os.listdir(FGSM_ADV_DIR) if f.endswith(".pth")
    ])

if RUN_BIM_ADV_TRAIN:
    bim_adv_paths = run_stage_with_monitor("bim_adv_train", train_all_bim_students)
else:
    bim_adv_paths = sorted([
        os.path.join(BIM_ADV_DIR, f) for f in os.listdir(BIM_ADV_DIR) if f.endswith(".pth")
    ])

if RUN_PGD_ADV_TRAIN:
    pgd_adv_paths = run_stage_with_monitor("pgd_adv_train", train_all_pgd_students)
else:
    pgd_adv_paths = sorted([
        os.path.join(PGD_ADV_DIR, f) for f in os.listdir(PGD_ADV_DIR) if f.endswith(".pth")
    ])

print("\nFGSM adv students:")
for p in fgsm_adv_paths:
    print(" ", p)

print("\nBIM adv students:")
for p in bim_adv_paths:
    print(" ", p)

print("\nPGD adv students:")
for p in pgd_adv_paths:
    print(" ", p)


==================== STAGE: fgsm_adv_train ====================
[FGSM] Resuming from: /content/drive/MyDrive/PES/project/checkpoints/fgsm/vanilla_student_01_fgsm_adv_resume.pt


/tmp/ipykernel_399/2283914661.py:61: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


Saved FGSM adv student: /content/drive/MyDrive/PES/project/models/vanStudents_adv/fgsm/vanilla_student_01_fgsm_adv.pth
Saved FGSM train history: /content/drive/MyDrive/PES/project/logs/fgsm/vanilla_student_01_fgsm_adv_train_history.csv
[FGSM] Resuming from: /content/drive/MyDrive/PES/project/checkpoints/fgsm/vanilla_student_02_fgsm_adv_resume.pt


/tmp/ipykernel_399/2283914661.py:61: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


Saved FGSM adv student: /content/drive/MyDrive/PES/project/models/vanStudents_adv/fgsm/vanilla_student_02_fgsm_adv.pth
Saved FGSM train history: /content/drive/MyDrive/PES/project/logs/fgsm/vanilla_student_02_fgsm_adv_train_history.csv
[FGSM] Resuming from: /content/drive/MyDrive/PES/project/checkpoints/fgsm/vanilla_student_03_fgsm_adv_resume.pt


...

Saved PGD adv student: /content/drive/MyDrive/PES/project/models/vanStudents_adv/pgd/vanilla_student_02_pgd_adv.pth
Saved PGD train history: /content/drive/MyDrive/PES/project/logs/pgd/vanilla_student_02_pgd_adv_train_history.csv
[PGD] Resuming from: /content/drive/MyDrive/PES/project/checkpoints/pgd/vanilla_student_03_pgd_adv_resume.pt


/tmp/ipykernel_399/2283914661.py:61: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


Saved PGD adv student: /content/drive/MyDrive/PES/project/models/vanStudents_adv/pgd/vanilla_student_03_pgd_adv.pth
Saved PGD train history: /content/drive/MyDrive/PES/project/logs/pgd/vanilla_student_03_pgd_adv_train_history.csv
[PGD] Resuming from: /content/drive/MyDrive/PES/project/checkpoints/pgd/vanilla_student_04_pgd_adv_resume.pt


/tmp/ipykernel_399/2283914661.py:61: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


Saved PGD adv student: /content/drive/MyDrive/PES/project/models/vanStudents_adv/pgd/vanilla_student_04_pgd_adv.pth
Saved PGD train history: /content/drive/MyDrive/PES/project/logs/pgd/vanilla_student_04_pgd_adv_train_history.csv
Stage resource summary:
{'n_samples': 9, 'sample_interval_sec': 2.0, 'max_cpu_ram_mb': 7281.125, 'max_gpu_mem_mb': 704.0, 'avg_gpu_util_percent': 0.0, 'gpu_active_percent_of_samples': 0.0, 'est_gpu_energy_j': 1087.4810384595394, 'stage_name': 'pgd_adv_train', 'stage_elapsed_min': 0.3041026552518209}

FGSM adv students:
  /content/drive/MyDrive/PES/project/models/vanStudents_adv/fgsm/vanilla_student_01_fgsm_adv.pth
  /content/drive/MyDrive/PES/project/models/vanStudents_adv/fgsm/vanilla_student_02_fgsm_adv.pth
  /content/drive/MyDrive/PES/project/models/vanStudents_adv/fgsm/vanilla_student_03_fgsm_adv.pth
  /content/drive/MyDrive/PES/project/models/vanStudents_adv/fgsm/vanilla_student_04_fgsm_adv.pth

BIM adv students:
  /content/drive/MyDrive/PES/project/model

In [ ]:
# resource summary

if os.path.exists(PIPELINE_RESOURCE_SUMMARY_CSV):
    pipeline_resource_df = pd.read_csv(PIPELINE_RESOURCE_SUMMARY_CSV)
    print("\nPipeline resource summary:")
    print(pipeline_resource_df)


Pipeline resource summary:
   n_samples  sample_interval_sec  max_cpu_ram_mb  max_gpu_mem_mb  \
0          5                  2.0     7191.812500           642.0   
1          2                  2.0     7233.855469           688.0   
2          1                  2.0     7249.429688           688.0   
3          1                  2.0     7249.613281           688.0   
4         12                  2.0     7288.765625           688.0   
5         11                  2.0     7279.585938           704.0   
6          9                  2.0     7279.609375           704.0   
7          9                  2.0     7281.125000           704.0   

   avg_gpu_util_percent  gpu_active_percent_of_samples  est_gpu_energy_j  \
0              0.600000                      20.000000        543.919067   
1             28.000000                      50.000000        168.249906   
2              0.000000                       0.000000               NaN   
3              0.000000                       

In [ ]:
def model_n_params(model: nn.Module):
    return sum(p.numel() for p in model.parameters())

def model_file_size_mb(path: str):
    if not os.path.exists(path):
        return None
    return os.path.getsize(path) / (1024.0 * 1024.0)

def append_df_to_csv(df, path):
    header = not os.path.exists(path)
    df.to_csv(path, mode="a", header=header, index=False)

MODEL_SIZES_NOV_CSV = os.path.join(BASE_MODEL_DIR, "appEnergyNov_model_sizes.csv")

In [ ]:
# reload adv students

# fgsm adv students
fgsm_adv_paths = sorted([
    os.path.join(FGSM_ADV_DIR, f)
    for f in os.listdir(FGSM_ADV_DIR)
    if f.endswith("_fgsm_adv.pth")
])

print("\nFGSM adv students found:", len(fgsm_adv_paths))
for p in fgsm_adv_paths:
    print("  ", p)

# bim adv students
bim_adv_paths = sorted([
    os.path.join(BIM_ADV_DIR, f)
    for f in os.listdir(BIM_ADV_DIR)
    if f.endswith("_bim_adv.pth")
])

print("\nBIM adv students found:", len(bim_adv_paths))
for p in bim_adv_paths:
    print("  ", p)

# pgd adv students
pgd_adv_paths = sorted([
    os.path.join(PGD_ADV_DIR, f)
    for f in os.listdir(PGD_ADV_DIR)
    if f.endswith("_pgd_adv.pth")
])

print("\nPGD adv students found:", len(pgd_adv_paths))
for p in pgd_adv_paths:
    print("  ", p)



# move to device
def load_adv_students(path_list):
    students = []
    for p in path_list:
        m = build_model().to(device)
        m.load_state_dict(torch.load(p, map_location=device))
        m.eval()
        students.append(m)
    return students

# load pools
fgsm_students = load_adv_students(fgsm_adv_paths)
bim_students  = load_adv_students(bim_adv_paths)
pgd_students  = load_adv_students(pgd_adv_paths)

fgsm_student_names = [os.path.basename(p).replace(".pth", "") for p in fgsm_adv_paths]
bim_student_names  = [os.path.basename(p).replace(".pth", "") for p in bim_adv_paths]
pgd_student_names  = [os.path.basename(p).replace(".pth", "") for p in pgd_adv_paths]

print("\nLoaded adv student pools:")
print(" FGSM:", len(fgsm_students))
print(" BIM :", len(bim_students))
print(" PGD :", len(pgd_students))


FGSM adv students found: 4
   /content/drive/MyDrive/PES/project/models/vanStudents_adv/fgsm/vanilla_student_01_fgsm_adv.pth
   /content/drive/MyDrive/PES/project/models/vanStudents_adv/fgsm/vanilla_student_02_fgsm_adv.pth
   /content/drive/MyDrive/PES/project/models/vanStudents_adv/fgsm/vanilla_student_03_fgsm_adv.pth
   /content/drive/MyDrive/PES/project/models/vanStudents_adv/fgsm/vanilla_student_04_fgsm_adv.pth

BIM adv students found: 4
   /content/drive/MyDrive/PES/project/models/vanStudents_adv/bim/vanilla_student_01_bim_adv.pth
   /content/drive/MyDrive/PES/project/models/vanStudents_adv/bim/vanilla_student_02_bim_adv.pth
   /content/drive/MyDrive/PES/project/models/vanStudents_adv/bim/vanilla_student_03_bim_adv.pth
   /content/drive/MyDrive/PES/project/models/vanStudents_adv/bim/vanilla_student_04_bim_adv.pth

PGD adv students found: 4
   /content/drive/MyDrive/PES/project/models/vanStudents_adv/pgd/vanilla_student_01_pgd_adv.pth
   /content/drive/MyDrive/PES/project/models/v

/tmp/ipykernel_399/2283914661.py:61: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)



Loaded adv student pools:
 FGSM: 4
 BIM : 4
 PGD : 4


In [ ]:
# evaluation

# eval helpers

def eval_clean_rmse(model, loader):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            p  = model(xb)
            ys.append(yb.cpu().numpy())
            ps.append(p.cpu().numpy())
    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return float(np.sqrt(mean_squared_error(ys.ravel(), ps.ravel())))
#added .ravel to flatten (encounter error dim 3 )
def eval_ensemble_rmse(models, loader):
    for m in models:
        m.eval()
    ys, ps = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = []
            for m in models:
                preds.append(m(xb))
            preds = torch.stack(preds, dim=0).mean(dim=0)
            ys.append(yb.cpu().numpy())
            ps.append(preds.cpu().numpy())
    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return float(np.sqrt(mean_squared_error(ys.ravel(), ps.ravel())))

def eval_pair_attack(defender, attacker, loader, attack_fn, atk_kwargs):
    defender.eval()
    attacker.eval()
    ys, ps = [], []
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        xa = attack_fn(attacker, xb, yb, **atk_kwargs)
        with torch.no_grad():
            p = defender(xa)
        ys.append(yb.cpu().numpy())
        ps.append(p.cpu().numpy())
    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return float(np.sqrt(mean_squared_error(ys.ravel(), ps.ravel())))

def eval_random_switch_attack(models, loader, attack_fn, atk_kwargs_func):

    #atk_kwargs_func: function() - dict with eps/alpha/iters already baked in

    for m in models:
        m.eval()
    ys, ps = [], []
    n_stu = len(models)

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)

        atk_model = random.choice(models)
        def_model = random.choice(models)

        atk_kwargs = atk_kwargs_func()

        xa = attack_fn(atk_model, xb, yb, **atk_kwargs)
        with torch.no_grad():
            p = def_model(xa)
        ys.append(yb.cpu().numpy())
        ps.append(p.cpu().numpy())

    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return float(np.sqrt(mean_squared_error(ys.ravel(), ps.ravel())))


In [ ]:
# diversity metrics
# weight + prediction

def flatten_params(model):
    return torch.cat([p.detach().cpu().view(-1) for p in model.parameters()])

def compute_weight_diversity(students, names, out_csv):
    vecs = [flatten_params(m) for m in students]
    k = len(vecs)
    rows = []
    for i, j in itertools.combinations(range(k), 2):
        d = torch.norm(vecs[i] - vecs[j], p=2).item()
        rows.append({
            "student_i": names[i],
            "student_j": names[j],
            "l2_distance": d,
        })
    df_w = pd.DataFrame(rows)
    df_w.to_csv(out_csv, index=False)
    print("Saved weight diversity to:", out_csv)
    return df_w

@torch.no_grad()
def compute_prediction_diversity(students, loader, out_csv):

    for m in students:
        m.eval()

    k = len(students)
    sum_pred  = None
    sum_pred2 = None
    count     = 0

    for xb, yb in loader:
        xb = xb.to(device)

        preds_stu = []
        for m in students:
            # each: (B, 1) -> flatten to (B,)
            p = m(xb).cpu().numpy().reshape(-1)
            preds_stu.append(p)

        # (k, B)
        preds_stu = np.stack(preds_stu, axis=0)

        # average over batch dimension -> (k,)
        preds_batch_mean = preds_stu.mean(axis=1)

        if sum_pred is None:
            sum_pred  = np.zeros_like(preds_batch_mean, dtype=np.float64)
            sum_pred2 = np.zeros_like(preds_batch_mean, dtype=np.float64)

        sum_pred  += preds_batch_mean
        sum_pred2 += preds_batch_mean ** 2
        count     += 1

    if count == 0:
        raise RuntimeError("compute_prediction_diversity: loader produced 0 batches")

    mean_pred  = sum_pred  / count          # (k,)
    mean_pred2 = sum_pred2 / count          # (k,)
    var_pred   = np.maximum(mean_pred2 - mean_pred**2, 0.0)

    df_p = pd.DataFrame({
        "student_idx": np.arange(k),
        "mean_pred":   mean_pred,
        "var_pred":    var_pred,
    })
    df_p.to_csv(out_csv, index=False)
    print("Saved prediction diversity to:", out_csv)
    return df_p

In [ ]:
# transferability matrix
# per epsilon, per attack type

def compute_transfer_matrix(
    students,
    names,
    loader,
    attack_fn,
    eps_list,
    extra_kwargs_func,
    out_csv,
    attack_label="rfgsm",
):
    rows = []

    # hb_flag, hb_thread = start_heartbeat(f"{attack_label}_transfer_matrix") # Removed heartbeat call
    start_time = time.time()

    try:
        for eps in eps_list:
            atk_kwargs = extra_kwargs_func(eps)
            print(f"\n[{attack_label.upper()}] Transfer matrix for eps={eps}...")
            for i, atk_name in enumerate(names):
                for j, def_name in enumerate(names):
                    atk_model = students[i]
                    def_model = students[j]
                    rmse_ij = eval_pair_attack(
                        defender=def_model,
                        attacker=atk_model,
                        loader=loader,
                        attack_fn=attack_fn,
                        atk_kwargs=atk_kwargs,
                    )
                    rows.append({
                        "attack": attack_label,
                        "epsilon": eps,
                        "attacker": atk_name,
                        "defender": def_name,
                        "rmse_adv": rmse_ij,
                    })
                    print(f"  atk={atk_name} - def={def_name} | eps={eps:.2f} | RMSE={rmse_ij:.5f}")
    finally:
        # stop_heartbeat(hb_flag, hb_thread) # Removed heartbeat call
        pass # Keep a pass for now

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    print(f"\nSaved {attack_label} transfer matrix to:", out_csv)
    return df

In [ ]:
# 30 run eval

def run_morphence_eval(
    attack_name,
    attack_fn,
    eps_list,
    students,
    results_csv,
    stats_csv,
    extra_kwargs_func,
    n_runs=30,
):
    """
    attack_name: "rfgsm", "bim", "pgd"
    attack_fn:   rfgsm, bim_attack, pgd_attack
    eps_list:    list of epsilons
    students:    list of adv-trained student models
    extra_kwargs_func: function(eps) -> dict of kwargs to pass to attack_fn
    """
    all_rows = []  # ["attack","epsilon","model","RMSE","run","seed"]

    # hb_flag, hb_thread = start_heartbeat(f"{attack_name}_morphence_eval") # Removed heartbeat call
    start_all = time.time()

    try:
        for run_i in range(1, n_runs + 1):
            run_start = time.time()
            seed_i = 3000 + run_i
            random.seed(seed_i)
            np.random.seed(seed_i)
            torch.manual_seed(seed_i)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(seed_i)

            # clean
            base_clean   = eval_clean_rmse(best_base, test_dl) # Changed from base_eval
            morph_clean  = eval_ensemble_rmse(students, test_dl)
            all_rows.append(["clean", None, "base",           base_clean,  run_i, seed_i])
            all_rows.append(["clean", None, "morph_ensemble", morph_clean, run_i, seed_i])

            print(f"[{attack_name.upper()} | Run {run_i:02d}] clean | "
                  f"base={base_clean:.5f} | morph_ens={morph_clean:.5f}")

            # attack
            for eps in eps_list:
                atk_kwargs = extra_kwargs_func(eps)

                # base white box (attacker = defender = base_eval)
                base_rmse = eval_pair_attack(
                    defender=best_base, # Changed from base_eval
                    attacker=best_base, # Changed from base_eval
                    loader=test_dl,
                    attack_fn=attack_fn,
                    atk_kwargs=atk_kwargs,
                )

                # random switching
                # attacker+defender random from pool
                def atk_kwargs_wrapper():
                    return atk_kwargs

                morph_rmse = eval_random_switch_attack(
                    models=students,
                    loader=test_dl,
                    attack_fn=attack_fn,
                    atk_kwargs_func=atk_kwargs_wrapper,
                )

                all_rows.append([attack_name, eps, "base",       base_rmse,  run_i, seed_i])
                all_rows.append([attack_name, eps, "morph_rand", morph_rmse, run_i, seed_i])

                print(
                    f"[{attack_name.upper()} | Run {run_i:02d}] eps={eps:.2f} | "
                    f"base={base_rmse:.5f} | morph_rand={morph_rmse:.5f}"
                )

            # incremental save
            df_runs = pd.DataFrame(
                all_rows,
                columns=["attack","epsilon","model","RMSE","run","seed"]
            )
            df_runs.to_csv(results_csv, index=False)

            stats = (
                df_runs
                .groupby(["attack","epsilon","model"])["RMSE"]
                .agg(mean="mean", std="std", n="count")
                .reset_index()
                .sort_values(["attack","epsilon","model"])
            )
            stats.to_csv(stats_csv, index=False)

            run_mins   = (time.time() - run_start) / 60.0
            total_mins = (time.time() - start_all) / 60.0
            print(
                f"[{attack_name.upper()}] Completed run {run_i}/{n_runs} | "
                f"run_time={run_mins:.2f} min | total_elapsed={total_mins:.2f} min",
                flush=True
            )

            if torch.cuda.is_available():
                torch.cuda.synchronize()
                torch.cuda.empty_cache()
            gc.collect()

    finally:
        # stop_heartbeat(hb_flag, hb_thread) # Removed heartbeat call
        pass # Keep a pass for now

    print(f"\nSaved {attack_name} runs to:", results_csv)
    print(f"Saved {attack_name} stats to:", stats_csv)
    return df_runs, stats

In [ ]:
#FGSM diversity + transferability + 30 run eval

def fgsm_kwargs(eps):
    return {"eps": eps, "alpha": FGSM_ALPHA}

df_fgsm_weight_div = pd.DataFrame()
df_fgsm_pred_div = pd.DataFrame()
df_fgsm_xfer_matrix = pd.DataFrame()
df_fgsm_runs = pd.DataFrame()
fgsm_stats = pd.DataFrame()

def _run_fgsm_eval():
    global df_fgsm_weight_div, df_fgsm_pred_div, df_fgsm_xfer_matrix, df_fgsm_runs, fgsm_stats

    if RUN_FGSM_EVAL:
        print("Running FGSM evaluation (or loading if files exist)...")

        # Weight Diversity
        if os.path.exists(FGSM_WEIGHT_DIVERSITY_CSV):
            df_fgsm_weight_div = pd.read_csv(FGSM_WEIGHT_DIVERSITY_CSV)
            print(f"Loaded FGSM weight diversity from: {FGSM_WEIGHT_DIVERSITY_CSV}")
        else:
            df_fgsm_weight_div = compute_weight_diversity(fgsm_students, fgsm_student_names, FGSM_WEIGHT_DIVERSITY_CSV)

        # Prediction Diversity
        if os.path.exists(FGSM_PRED_DIVERSITY_CSV):
            df_fgsm_pred_div = pd.read_csv(FGSM_PRED_DIVERSITY_CSV)
            print(f"Loaded FGSM prediction diversity from: {FGSM_PRED_DIVERSITY_CSV}")
        else:
            df_fgsm_pred_div = compute_prediction_diversity(fgsm_students, test_dl, FGSM_PRED_DIVERSITY_CSV)

        # Transfer Matrix
        if os.path.exists(FGSM_XFER_MATRIX_CSV):
            df_fgsm_xfer_matrix = pd.read_csv(FGSM_XFER_MATRIX_CSV)
            print(f"Loaded FGSM transfer matrix from: {FGSM_XFER_MATRIX_CSV}")
        else:
            df_fgsm_xfer_matrix = compute_transfer_matrix(
                students=fgsm_students,
                names=fgsm_student_names,
                loader=test_dl,
                attack_fn=rfgsm_attack,
                eps_list=FGSM_TRAIN_EPS_LIST,
                extra_kwargs_func=fgsm_kwargs,
                out_csv=FGSM_XFER_MATRIX_CSV,
                attack_label="rfgsm",
            )

        # 30-run Evaluation
        if os.path.exists(FGSM_RUNS_CSV) and os.path.exists(FGSM_STATS_CSV):
            df_fgsm_runs = pd.read_csv(FGSM_RUNS_CSV)
            fgsm_stats = pd.read_csv(FGSM_STATS_CSV)
            print(f"Loaded FGSM 30-run results from: {FGSM_RUNS_CSV} and {FGSM_STATS_CSV}")
        else:
            df_fgsm_runs, fgsm_stats = run_morphence_eval(
                attack_name="rfgsm",
                attack_fn=rfgsm_attack,
                eps_list=FGSM_TRAIN_EPS_LIST,
                students=fgsm_students,
                results_csv=FGSM_RUNS_CSV,
                stats_csv=FGSM_STATS_CSV,
                extra_kwargs_func=fgsm_kwargs,
                n_runs=30,
            )
    else:
        print("Skipping FGSM evaluation. Loading results from disk.")
        df_fgsm_weight_div = pd.read_csv(FGSM_WEIGHT_DIVERSITY_CSV) if os.path.exists(FGSM_WEIGHT_DIVERSITY_CSV) else pd.DataFrame()
        df_fgsm_pred_div = pd.read_csv(FGSM_PRED_DIVERSITY_CSV) if os.path.exists(FGSM_PRED_DIVERSITY_CSV) else pd.DataFrame()
        df_fgsm_xfer_matrix = pd.read_csv(FGSM_XFER_MATRIX_CSV) if os.path.exists(FGSM_XFER_MATRIX_CSV) else pd.DataFrame()
        df_fgsm_runs = pd.read_csv(FGSM_RUNS_CSV) if os.path.exists(FGSM_RUNS_CSV) else pd.DataFrame()
        fgsm_stats = pd.read_csv(FGSM_STATS_CSV) if os.path.exists(FGSM_STATS_CSV) else pd.DataFrame()

run_stage_with_monitor("fgsm_eval", _run_fgsm_eval)


Running FGSM evaluation (or loading if files exist)...
Loaded FGSM weight diversity from: /content/drive/MyDrive/PES/project/reports/fgsm_results/fgsm_weight_diversity.csv
Loaded FGSM prediction diversity from: /content/drive/MyDrive/PES/project/reports/fgsm_results/fgsm_prediction_diversity.csv
Loaded FGSM transfer matrix from: /content/drive/MyDrive/PES/project/reports/fgsm_results/fgsm_transfer_matrix.csv
Loaded FGSM 30-run results from: /content/drive/MyDrive/PES/project/reports/fgsm_results/fgsm_morphence_runs.csv and /content/drive/MyDrive/PES/project/reports/fgsm_results/fgsm_morphence_stats.csv


In [ ]:
#BIM diversity + transferability + 30 run eval

def bim_kwargs(eps):
    return {
        "eps": eps,
        "alpha": eps / BIM_ITERS,
        "iters": BIM_ITERS,
        "random_start": True,
    }

# Initialize DataFrames to ensure they always exist
df_bim_weight_div = pd.DataFrame()
df_bim_pred_div = pd.DataFrame()
df_bim_xfer_matrix = pd.DataFrame()
df_bim_runs = pd.DataFrame()
bim_stats = pd.DataFrame()

def _run_bim_eval():
    global df_bim_weight_div, df_bim_pred_div, df_bim_xfer_matrix, df_bim_runs, bim_stats

    if RUN_BIM_EVAL:
        print("Running BIM evaluation (or loading if files exist)...")

        # Weight Diversity
        if os.path.exists(BIM_WEIGHT_DIVERSITY_CSV):
            df_bim_weight_div = pd.read_csv(BIM_WEIGHT_DIVERSITY_CSV)
            print(f"Loaded BIM weight diversity from: {BIM_WEIGHT_DIVERSITY_CSV}")
        else:
            df_bim_weight_div = compute_weight_diversity(bim_students, bim_student_names, BIM_WEIGHT_DIVERSITY_CSV)

        # Prediction Diversity
        if os.path.exists(BIM_PRED_DIVERSITY_CSV):
            df_bim_pred_div = pd.read_csv(BIM_PRED_DIVERSITY_CSV)
            print(f"Loaded BIM prediction diversity from: {BIM_PRED_DIVERSITY_CSV}")
        else:
            df_bim_pred_div = compute_prediction_diversity(bim_students, test_dl, BIM_PRED_DIVERSITY_CSV)

        # Transfer Matrix
        if os.path.exists(BIM_XFER_MATRIX_CSV):
            df_bim_xfer_matrix = pd.read_csv(BIM_XFER_MATRIX_CSV)
            print(f"Loaded BIM transfer matrix from: {BIM_XFER_MATRIX_CSV}")
        else:
            df_bim_xfer_matrix = compute_transfer_matrix(
                students=bim_students,
                names=bim_student_names,
                loader=test_dl,
                attack_fn=bim_attack,
                eps_list=BIM_TRAIN_EPS_LIST,
                extra_kwargs_func=bim_kwargs,
                out_csv=BIM_XFER_MATRIX_CSV,
                attack_label="bim",
            )

        # 30-run Evaluation
        if os.path.exists(BIM_RUNS_CSV) and os.path.exists(BIM_STATS_CSV):
            df_bim_runs = pd.read_csv(BIM_RUNS_CSV)
            bim_stats = pd.read_csv(BIM_STATS_CSV)
            print(f"Loaded BIM 30-run results from: {BIM_RUNS_CSV} and {BIM_STATS_CSV}")
        else:
            df_bim_runs, bim_stats = run_morphence_eval(
                attack_name="bim",
                attack_fn=bim_attack,
                eps_list=BIM_TRAIN_EPS_LIST,
                students=bim_students,
                results_csv=BIM_RUNS_CSV,
                stats_csv=BIM_STATS_CSV,
                extra_kwargs_func=bim_kwargs,
                n_runs=30,
            )
    else:
        print("Skipping BIM evaluation. Loading results from disk.")
        df_bim_weight_div = pd.read_csv(BIM_WEIGHT_DIVERSITY_CSV) if os.path.exists(BIM_WEIGHT_DIVERSITY_CSV) else pd.DataFrame()
        df_bim_pred_div = pd.read_csv(BIM_PRED_DIVERSITY_CSV) if os.path.exists(BIM_PRED_DIVERSITY_CSV) else pd.DataFrame()
        df_bim_xfer_matrix = pd.read_csv(BIM_XFER_MATRIX_CSV) if os.path.exists(BIM_XFER_MATRIX_CSV) else pd.DataFrame()
        df_bim_runs = pd.read_csv(BIM_RUNS_CSV) if os.path.exists(BIM_RUNS_CSV) else pd.DataFrame()
        bim_stats = pd.read_csv(BIM_STATS_CSV) if os.path.exists(BIM_STATS_CSV) else pd.DataFrame()

run_stage_with_monitor("bim_eval", _run_bim_eval)


Running BIM evaluation (or loading if files exist)...
Loaded BIM weight diversity from: /content/drive/MyDrive/PES/project/reports/bim_results/bim_weight_diversity.csv
Loaded BIM prediction diversity from: /content/drive/MyDrive/PES/project/reports/bim_results/bim_prediction_diversity.csv
Loaded BIM transfer matrix from: /content/drive/MyDrive/PES/project/reports/bim_results/bim_transfer_matrix.csv
Loaded BIM 30-run results from: /content/drive/MyDrive/PES/project/reports/bim_results/bim_morphence_runs.csv and /content/drive/MyDrive/PES/project/reports/bim_results/bim_morphence_stats.csv


In [ ]:
print("\nPGD diversity + transferability + 30 run eval")

def pgd_kwargs(eps):
    return {
        "eps": eps,
        "alpha": eps / PGD_ITERS,
        "iters": PGD_ITERS,
        "random_start": True,
    }
df_pgd_weight_div = pd.DataFrame()
df_pgd_pred_div = pd.DataFrame()
df_pgd_xfer_matrix = pd.DataFrame()
df_pgd_runs = pd.DataFrame()
pgd_stats = pd.DataFrame()

def _run_pgd_eval():
    global df_pgd_weight_div, df_pgd_pred_div, df_pgd_xfer_matrix, df_pgd_runs, pgd_stats

    if RUN_PGD_EVAL:
        print("Running PGD evaluation (or loading if files exist)...")

        # Weight Diversity
        if os.path.exists(PGD_WEIGHT_DIVERSITY_CSV):
            df_pgd_weight_div = pd.read_csv(PGD_WEIGHT_DIVERSITY_CSV)
            print(f"Loaded PGD weight diversity from: {PGD_WEIGHT_DIVERSITY_CSV}")
        else:
            df_pgd_weight_div = compute_weight_diversity(pgd_students, pgd_student_names, PGD_WEIGHT_DIVERSITY_CSV)

        # Prediction Diversity
        if os.path.exists(PGD_PRED_DIVERSITY_CSV):
            df_pgd_pred_div = pd.read_csv(PGD_PRED_DIVERSITY_CSV)
            print(f"Loaded PGD prediction diversity from: {PGD_PRED_DIVERSITY_CSV}")
        else:
            df_pgd_pred_div = compute_prediction_diversity(pgd_students, test_dl, PGD_PRED_DIVERSITY_CSV)

        # Transfer Matrix
        if os.path.exists(PGD_XFER_MATRIX_CSV):
            df_pgd_xfer_matrix = pd.read_csv(PGD_XFER_MATRIX_CSV)
            print(f"Loaded PGD transfer matrix from: {PGD_XFER_MATRIX_CSV}")
        else:
            df_pgd_xfer_matrix = compute_transfer_matrix(
                students=pgd_students,
                names=pgd_student_names,
                loader=test_dl,
                attack_fn=pgd_attack,
                eps_list=PGD_TRAIN_EPS_LIST,
                extra_kwargs_func=pgd_kwargs,
                out_csv=PGD_XFER_MATRIX_CSV,
                attack_label="pgd",
            )

        # 30-run Evaluation
        if os.path.exists(PGD_RUNS_CSV) and os.path.exists(PGD_STATS_CSV):
            df_pgd_runs = pd.read_csv(PGD_RUNS_CSV)
            pgd_stats = pd.read_csv(PGD_STATS_CSV)
            print(f"Loaded PGD 30-run results from: {PGD_RUNS_CSV} and {PGD_STATS_CSV}")
        else:
            df_pgd_runs, pgd_stats = run_morphence_eval(
                attack_name="pgd",
                attack_fn=pgd_attack,
                eps_list=PGD_TRAIN_EPS_LIST,
                students=pgd_students,
                results_csv=PGD_RUNS_CSV,
                stats_csv=PGD_STATS_CSV,
                extra_kwargs_func=pgd_kwargs,
                n_runs=30,
            )
    else:
        print("Skipping PGD evaluation. Loading results from disk..")
        df_pgd_weight_div = pd.read_csv(PGD_WEIGHT_DIVERSITY_CSV) if os.path.exists(PGD_WEIGHT_DIVERSITY_CSV) else pd.DataFrame()
        df_pgd_pred_div = pd.read_csv(PGD_PRED_DIVERSITY_CSV) if os.path.exists(PGD_PRED_DIVERSITY_CSV) else pd.DataFrame()
        df_pgd_xfer_matrix = pd.read_csv(PGD_XFER_MATRIX_CSV) if os.path.exists(PGD_XFER_MATRIX_CSV) else pd.DataFrame()
        df_pgd_runs = pd.read_csv(PGD_RUNS_CSV) if os.path.exists(PGD_RUNS_CSV) else pd.DataFrame()
        pgd_stats = pd.read_csv(PGD_STATS_CSV) if os.path.exists(PGD_STATS_CSV) else pd.DataFrame()

run_stage_with_monitor("pgd_eval", _run_pgd_eval)



PGD diversity + transferability + 30 run eval
Running PGD evaluation (or loading if files exist)...
Loaded PGD weight diversity from: /content/drive/MyDrive/PES/project/reports/pgd_results/pgd_weight_diversity.csv
Loaded PGD prediction diversity from: /content/drive/MyDrive/PES/project/reports/pgd_results/pgd_prediction_diversity.csv
Loaded PGD transfer matrix from: /content/drive/MyDrive/PES/project/reports/pgd_results/pgd_transfer_matrix.csv
Loaded PGD 30-run results from: /content/drive/MyDrive/PES/project/reports/pgd_results/pgd_morphence_runs.csv and /content/drive/MyDrive/PES/project/reports/pgd_results/pgd_morphence_stats.csv


In [ ]:
import os
import pandas as pd
import numpy as np

# config for VANILLA attacks
ATTACK_CONFIGS_VANILLA = [
    dict(
        name="FGSM",
        prefix="fgsm",
        attack_key="rfgsm",   # column value in fgsm_morphence_runs.csv
        runs_csv=FGSM_RUNS_CSV,
        results_dir=FGSM_RESULTS_DIR,
    ),
    dict(
        name="BIM",
        prefix="bim",
        attack_key="bim",     # column value in bim_morphence_runs.csv
        runs_csv=BIM_RUNS_CSV,
        results_dir=BIM_RESULTS_DIR,
    ),
    dict(
        name="PGD",
        prefix="pgd",
        attack_key="pgd",     # column value in pgd_morphence_runs.csv
        runs_csv=PGD_RUNS_CSV,
        results_dir=PGD_RESULTS_DIR,
    ),
]

# compute stats for one config
def compute_stats_for_attack(cfg):
    name        = cfg["name"]
    prefix      = cfg["prefix"]
    attack_key  = cfg["attack_key"]
    runs_csv    = cfg["runs_csv"]
    results_dir = cfg["results_dir"]

    os.makedirs(results_dir, exist_ok=True)

    print("\n==============================================")
    print(f"Processing {name} runs")
    print(f"Runs CSV: {runs_csv}")
    print("==============================================")

    if not os.path.exists(runs_csv):
        print(f"Runs CSV does not exist yet: {runs_csv}")
        return

    # load runs CSV
    df_runs = pd.read_csv(runs_csv)

    # clean stats (base vs morph_ensemble)
    df_clean = df_runs[df_runs["attack"] == "clean"].copy()

    df_clean["model"] = pd.Categorical(
        df_clean["model"],
        categories=["base", "morph_ensemble"],
        ordered=True,
    )

    clean_stats = (
        df_clean
        .groupby("model", observed=False)["RMSE"]
        .agg(
            mean   ="mean",
            std    ="std",
            min    ="min",
            q1     =lambda s: s.quantile(0.25),
            median ="median",
            q3     =lambda s: s.quantile(0.75),
            max    ="max",
            n      ="count",
        )
        .reset_index()
        .sort_values("model")
    )

    print(f"\n[{name} | Clean] stats:")
    print(clean_stats.to_string(index=False))

    clean_csv = os.path.join(results_dir, f"{prefix}_clean_stats.csv")
    clean_stats.to_csv(clean_csv, index=False)
    print(f"Saved clean stats to: {clean_csv}")

    # attack stats (base vs morph_rand)
    df_attack = df_runs[df_runs["attack"] == attack_key].copy()

    def model_attack_stats(df, model_name):
        st = (
            df[df["model"] == model_name]
            .groupby("epsilon", observed=False)["RMSE"]
            .agg(
                mean   ="mean",
                std    ="std",
                min    ="min",
                q1     =lambda s: s.quantile(0.25),
                median ="median",
                q3     =lambda s: s.quantile(0.75),
                max    ="max",
                n      ="count",
            )
            .reset_index()
            .sort_values("epsilon")
        )

        print(f"\n[{name} | {attack_key}] {model_name} stats:")
        print(st.to_string(index=False))

        out_csv = os.path.join(
            results_dir, f"{prefix}_{attack_key}_{model_name}_stats.csv"
        )
        st.to_csv(out_csv, index=False)
        print(f"Saved: {out_csv}")

    model_attack_stats(df_attack, "base")
    model_attack_stats(df_attack, "morph_rand")

# run for FGSM, BIM, PGD
for cfg in ATTACK_CONFIGS_VANILLA:
    compute_stats_for_attack(cfg)



Processing FGSM runs
Runs CSV: /content/drive/MyDrive/PES/project/reports/fgsm_results/fgsm_morphence_runs.csv

[FGSM | Clean] stats:
         model     mean  std      min       q1   median       q3      max  n
          base 0.079786  0.0 0.079786 0.079786 0.079786 0.079786 0.079786  4
morph_ensemble 0.076196  0.0 0.076196 0.076196 0.076196 0.076196 0.076196  4
Saved clean stats to: /content/drive/MyDrive/PES/project/reports/fgsm_results/fgsm_clean_stats.csv

[FGSM | rfgsm] base stats:
 epsilon     mean      std      min       q1   median       q3      max  n
     0.1 0.103979 0.000029 0.103952 0.103962 0.103973 0.103989 0.104018  4
     0.2 0.104298 0.000042 0.104262 0.104263 0.104290 0.104324 0.104349  4
     0.3 0.104906 0.000045 0.104874 0.104877 0.104888 0.104917 0.104971  4
     0.5 0.106699 0.000103 0.106599 0.106635 0.106681 0.106745 0.106836  4
Saved: /content/drive/MyDrive/PES/project/reports/fgsm_results/fgsm_rfgsm_base_stats.csv

[FGSM | rfgsm] morph_rand stats:
 epsilon 

In [ ]:
# novel student definition:
# student 05  -  encoder self attention weights only
#                includes:
#                   encoder.layers.*.self_attn.in_proj_weight
#                   encoder.layers.*.self_attn.in_proj_bias
#                   encoder.layers.*.self_attn.out_proj.weight
#                   encoder.layers.*.self_attn.out_proj.bias
#
# student 06  -  encoder feed forward layers (MLP) only
#                includes:
#                   encoder.layers.*.linear1.weight / bias
#                   encoder.layers.*.linear2.weight / bias
#
# student 07  -  encoder LayerNorms only
#                includes:
#                   encoder.layers.*.norm1.weight / bias
#                   encoder.layers.*.norm2.weight / bias
#
# student 08  -  input projection + prediction head only
#                includes:
#                   in_proj.weight / bias
#                   head.(0).weight  (LayerNorm)
#                   head.(0).bias
#                   head.(1).weight  (final linear regressor)
#                   head.(1).bias

def is_enc_self_attn_param(name: str) -> bool:
    return name.startswith("encoder.layers.") and ".self_attn." in name

def is_ffn_param(name: str) -> bool:
    if not name.startswith("encoder.layers."):
        return False
    return (".linear1." in name) or (".linear2." in name)

def is_norm_param(name: str) -> bool:
    return name.startswith("encoder.layers.") and ".norm" in name

def is_inproj_head_param(name: str) -> bool:
    return name.startswith("in_proj") or name.startswith("head")

novel_student_specs = [
    dict(
        idx=5,
        name="student_05_enc_attn",
        selector=is_enc_self_attn_param,
        noise_scale=1e-3,
    ),
    dict(
        idx=6,
        name="student_06_ffn",
        selector=is_ffn_param,
        noise_scale=1e-3,
    ),
    dict(
        idx=7,
        name="student_07_norms",
        selector=is_norm_param,
        noise_scale=1e-3,
    ),
    dict(
        idx=8,
        name="student_08_inproj_head",
        selector=is_inproj_head_param,
        noise_scale=1e-3,
    )
]

In [ ]:
# generate 4 novel students
novel_student_paths = []

print("\n[Appliances Energy dataset] creating novel students from best_base")

for spec in novel_student_specs:
    idx      = spec["idx"]
    name     = spec["name"]
    selector = spec["selector"]
    sigma    = spec["noise_scale"]

    print(f"  [Novel Student {idx}] {name} | sigma={sigma}")

    # copy the trained best base
    st = copy.deepcopy(best_base)

    # perturb only selected parameters
    with torch.no_grad():
        for pname, p in st.named_parameters():
            if selector(pname):
                p.add_(sigma * torch.randn_like(p))

    # save to disk
    out_path = os.path.join(NOV_DIR, f"{name}.pth")
    torch.save(st.state_dict(), out_path)
    novel_student_paths.append((idx, name, out_path))
    print("    saved to:", out_path)

    # log file size
    size_mb = model_file_size_mb(out_path)
    df_ms = pd.DataFrame([{
        "student_idx": idx,
        "model_name": name,
        "path": out_path,
        "params": model_n_params(best_base),
        "size_mb": size_mb,
    }])
    append_df_to_csv(df_ms, MODEL_SIZES_NOV_CSV)

# sort by index
novel_student_paths = sorted(novel_student_paths, key=lambda x: x[0])

print("\n[Appliances Energy] novel students generated:")
for idx, name, path in novel_student_paths:
    print(f"  {idx}: {name} - {path}")


[Appliances Energy dataset] creating novel students from best_base
  [Novel Student 5] student_05_enc_attn | sigma=0.001
    saved to: /content/drive/MyDrive/PES/project/models/novStudents/student_05_enc_attn.pth
  [Novel Student 6] student_06_ffn | sigma=0.001
    saved to: /content/drive/MyDrive/PES/project/models/novStudents/student_06_ffn.pth
  [Novel Student 7] student_07_norms | sigma=0.001
    saved to: /content/drive/MyDrive/PES/project/models/novStudents/student_07_norms.pth
  [Novel Student 8] student_08_inproj_head | sigma=0.001
    saved to: /content/drive/MyDrive/PES/project/models/novStudents/student_08_inproj_head.pth

[Appliances Energy] novel students generated:
  5: student_05_enc_attn - /content/drive/MyDrive/PES/project/models/novStudents/student_05_enc_attn.pth
  6: student_06_ffn - /content/drive/MyDrive/PES/project/models/novStudents/student_06_ffn.pth
  7: student_07_norms - /content/drive/MyDrive/PES/project/models/novStudents/student_07_norms.pth
  8: student

In [ ]:
# adversarial training for novel students

FGSM_NOV_ADV_DIR = os.path.join(NOV_ADV_DIR, "fgsm")
BIM_NOV_ADV_DIR  = os.path.join(NOV_ADV_DIR, "bim")
PGD_NOV_ADV_DIR  = os.path.join(NOV_ADV_DIR, "pgd")

FGSM_NOV_LOGS_DIR = os.path.join(LOGS_ROOT, "fgsm_nov")
BIM_NOV_LOGS_DIR  = os.path.join(LOGS_ROOT, "bim_nov")
PGD_NOV_LOGS_DIR  = os.path.join(LOGS_ROOT, "pgd_nov")

FGSM_NOV_CKPT_DIR = os.path.join(CHECKPOINT_ROOT, "fgsm_nov")
BIM_NOV_CKPT_DIR  = os.path.join(CHECKPOINT_ROOT, "bim_nov")
PGD_NOV_CKPT_DIR  = os.path.join(CHECKPOINT_ROOT, "pgd_nov")

for d in [FGSM_NOV_ADV_DIR, BIM_NOV_ADV_DIR, PGD_NOV_ADV_DIR,
          FGSM_NOV_LOGS_DIR, BIM_NOV_LOGS_DIR, PGD_NOV_LOGS_DIR,
          FGSM_NOV_CKPT_DIR, BIM_NOV_CKPT_DIR, PGD_NOV_CKPT_DIR]:
    os.makedirs(d, exist_ok=True)

def train_all_fgsm_novel_students():
    outs = []
    for _, name, sp in novel_student_paths:
        out = train_student_adv_resumable(
            student_path=sp,
            attack_name="FGSM-Nov",
            attack_fn=rfgsm_attack,
            attack_out_dir=FGSM_NOV_ADV_DIR,
            attack_logs_dir=FGSM_NOV_LOGS_DIR,
            attack_ckpt_dir=FGSM_NOV_CKPT_DIR,
            out_suffix="_fgsm_adv",
            epochs=FGSM_EPOCHS,
            lr=LR_STUDENT,
            eps_list=FGSM_TRAIN_EPS_LIST,
            q=Q_LOSS_Q,
            attack_kwargs_fn=lambda eps: {"eps": eps, "alpha": FGSM_ALPHA}
        )
        outs.append(out)
    return outs

def train_all_bim_novel_students():
    outs = []
    for _, name, sp in novel_student_paths:
        out = train_student_adv_resumable(
            student_path=sp,
            attack_name="BIM-Nov",
            attack_fn=bim_attack,
            attack_out_dir=BIM_NOV_ADV_DIR,
            attack_logs_dir=BIM_NOV_LOGS_DIR,
            attack_ckpt_dir=BIM_NOV_CKPT_DIR,
            out_suffix="_bim_adv",
            epochs=BIM_EPOCHS,
            lr=LR_STUDENT,
            eps_list=BIM_TRAIN_EPS_LIST,
            q=Q_LOSS_Q,
            attack_kwargs_fn=lambda eps: {
                "eps": eps,
                "alpha": eps / max(1, BIM_ITERS),
                "iters": BIM_ITERS,
                "random_start": False
            }
        )
        outs.append(out)
    return outs

def train_all_pgd_novel_students():
    outs = []
    for _, name, sp in novel_student_paths:
        out = train_student_adv_resumable(
            student_path=sp,
            attack_name="PGD-Nov",
            attack_fn=pgd_attack,
            attack_out_dir=PGD_NOV_ADV_DIR,
            attack_logs_dir=PGD_NOV_LOGS_DIR,
            attack_ckpt_dir=PGD_NOV_CKPT_DIR,
            out_suffix="_pgd_adv",
            epochs=PGD_EPOCHS,
            lr=LR_STUDENT,
            eps_list=PGD_TRAIN_EPS_LIST,
            q=Q_LOSS_Q,
            attack_kwargs_fn=lambda eps: {
                "eps": eps,
                "alpha": eps / max(1, PGD_ITERS),
                "iters": PGD_ITERS,
                "random_start": True
            }
        )
        outs.append(out)
    return outs

if RUN_FGSM_ADV_TRAIN:
    fgsm_nov_adv_paths = run_stage_with_monitor("fgsm_nov_adv_train", train_all_fgsm_novel_students)
else:
    fgsm_nov_adv_paths = sorted([
        os.path.join(FGSM_NOV_ADV_DIR, f) for f in os.listdir(FGSM_NOV_ADV_DIR) if f.endswith(".pth")
    ])

if RUN_BIM_ADV_TRAIN:
    bim_nov_adv_paths = run_stage_with_monitor("bim_nov_adv_train", train_all_bim_novel_students)
else:
    bim_nov_adv_paths = sorted([
        os.path.join(BIM_NOV_ADV_DIR, f) for f in os.listdir(BIM_NOV_ADV_DIR) if f.endswith(".pth")
    ])

if RUN_PGD_ADV_TRAIN:
    pgd_nov_adv_paths = run_stage_with_monitor("pgd_nov_adv_train", train_all_pgd_novel_students)
else:
    pgd_nov_adv_paths = sorted([
        os.path.join(PGD_NOV_ADV_DIR, f) for f in os.listdir(PGD_NOV_ADV_DIR) if f.endswith(".pth")
    ])

print("\nFGSM novel adv students:")
for p in fgsm_nov_adv_paths:
    print(" ", p)

print("\nBIM novel adv students:")
for p in bim_nov_adv_paths:
    print(" ", p)

print("\nPGD novel adv students:")
for p in pgd_nov_adv_paths:
    print(" ", p)



==================== STAGE: fgsm_nov_adv_train ====================
[FGSM-Nov] Resuming from: /content/drive/MyDrive/PES/project/checkpoints/fgsm_nov/student_05_enc_attn_fgsm_adv_resume.pt


/tmp/ipykernel_399/2283914661.py:61: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


Saved FGSM-Nov adv student: /content/drive/MyDrive/PES/project/models/novStudents_adv/fgsm/student_05_enc_attn_fgsm_adv.pth
Saved FGSM-Nov train history: /content/drive/MyDrive/PES/project/logs/fgsm_nov/student_05_enc_attn_fgsm_adv_train_history.csv
[FGSM-Nov] Resuming from: /content/drive/MyDrive/PES/project/checkpoints/fgsm_nov/student_06_ffn_fgsm_adv_resume.pt


/tmp/ipykernel_399/2283914661.py:61: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


Saved FGSM-Nov adv student: /content/drive/MyDrive/PES/project/models/novStudents_adv/fgsm/student_06_ffn_fgsm_adv.pth
Saved FGSM-Nov train history: /content/drive/MyDrive/PES/project/logs/fgsm_nov/student_06_ffn_fgsm_adv_train_history.csv
[FGSM-Nov] Resuming from: /content/drive/MyDrive/PES/project/checkpoints/fgsm_nov/student_07_norms_fgsm_adv_resume.pt


...

Saved PGD-Nov adv student: /content/drive/MyDrive/PES/project/models/novStudents_adv/pgd/student_06_ffn_pgd_adv.pth
Saved PGD-Nov train history: /content/drive/MyDrive/PES/project/logs/pgd_nov/student_06_ffn_pgd_adv_train_history.csv
[PGD-Nov] Resuming from: /content/drive/MyDrive/PES/project/checkpoints/pgd_nov/student_07_norms_pgd_adv_resume.pt


/tmp/ipykernel_399/2283914661.py:61: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


Saved PGD-Nov adv student: /content/drive/MyDrive/PES/project/models/novStudents_adv/pgd/student_07_norms_pgd_adv.pth
Saved PGD-Nov train history: /content/drive/MyDrive/PES/project/logs/pgd_nov/student_07_norms_pgd_adv_train_history.csv
[PGD-Nov] Resuming from: /content/drive/MyDrive/PES/project/checkpoints/pgd_nov/student_08_inproj_head_pgd_adv_resume.pt


/tmp/ipykernel_399/2283914661.py:61: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


Saved PGD-Nov adv student: /content/drive/MyDrive/PES/project/models/novStudents_adv/pgd/student_08_inproj_head_pgd_adv.pth
Saved PGD-Nov train history: /content/drive/MyDrive/PES/project/logs/pgd_nov/student_08_inproj_head_pgd_adv_train_history.csv
Stage resource summary:
{'n_samples': 8, 'sample_interval_sec': 2.0, 'max_cpu_ram_mb': 7390.1953125, 'max_gpu_mem_mb': 912.0, 'avg_gpu_util_percent': 0.125, 'gpu_active_percent_of_samples': 12.5, 'est_gpu_energy_j': 951.1653277945518, 'stage_name': 'pgd_nov_adv_train', 'stage_elapsed_min': 0.2703345537185669}

FGSM novel adv students:
  /content/drive/MyDrive/PES/project/models/novStudents_adv/fgsm/student_05_enc_attn_fgsm_adv.pth
  /content/drive/MyDrive/PES/project/models/novStudents_adv/fgsm/student_06_ffn_fgsm_adv.pth
  /content/drive/MyDrive/PES/project/models/novStudents_adv/fgsm/student_07_norms_fgsm_adv.pth
  /content/drive/MyDrive/PES/project/models/novStudents_adv/fgsm/student_08_inproj_head_fgsm_adv.pth

BIM novel adv students:
 

In [ ]:
APPENERGYNOV_DIR = os.path.join(PROJECT_ROOT, "appEnergyNov")

# Setup paths for novel student evaluations
FGSM_NOV_RESULTS_DIR = os.path.join(APPENERGYNOV_DIR, "fgsm", "results")
BIM_NOV_RESULTS_DIR  = os.path.join(APPENERGYNOV_DIR, "bim", "results")
PGD_NOV_RESULTS_DIR  = os.path.join(APPENERGYNOV_DIR, "pgd", "results")

for d in [FGSM_NOV_RESULTS_DIR, BIM_NOV_RESULTS_DIR, PGD_NOV_RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

FGSM_NOV_WEIGHT_DIVERSITY_CSV = os.path.join(FGSM_NOV_RESULTS_DIR, "nov_fgsm_weight_diversity.csv")
FGSM_NOV_PRED_DIVERSITY_CSV   = os.path.join(FGSM_NOV_RESULTS_DIR, "nov_fgsm_prediction_diversity.csv")
FGSM_NOV_XFER_MATRIX_CSV      = os.path.join(FGSM_NOV_RESULTS_DIR, "nov_fgsm_transfer_matrix.csv")
FGSM_NOV_STATS_CSV            = os.path.join(FGSM_NOV_RESULTS_DIR, "nov_fgsm_morphence_stats.csv")
FGSM_NOV_RUNS_CSV             = os.path.join(FGSM_NOV_RESULTS_DIR, "nov_fgsm_morphence_runs.csv")

BIM_NOV_WEIGHT_DIVERSITY_CSV  = os.path.join(BIM_NOV_RESULTS_DIR, "nov_bim_weight_diversity.csv")
BIM_NOV_PRED_DIVERSITY_CSV    = os.path.join(BIM_NOV_RESULTS_DIR, "nov_bim_prediction_diversity.csv")
BIM_NOV_XFER_MATRIX_CSV       = os.path.join(BIM_NOV_RESULTS_DIR, "nov_bim_transfer_matrix.csv")
BIM_NOV_STATS_CSV             = os.path.join(BIM_NOV_RESULTS_DIR, "nov_bim_morphence_stats.csv")
BIM_NOV_RUNS_CSV              = os.path.join(BIM_NOV_RESULTS_DIR, "nov_bim_morphence_runs.csv")

PGD_NOV_WEIGHT_DIVERSITY_CSV  = os.path.join(PGD_NOV_RESULTS_DIR, "nov_pgd_weight_diversity.csv")
PGD_NOV_PRED_DIVERSITY_CSV    = os.path.join(PGD_NOV_RESULTS_DIR, "nov_pgd_prediction_diversity.csv")
PGD_NOV_XFER_MATRIX_CSV       = os.path.join(PGD_NOV_RESULTS_DIR, "nov_pgd_transfer_matrix.csv")
PGD_NOV_STATS_CSV             = os.path.join(PGD_NOV_RESULTS_DIR, "nov_pgd_morphence_stats.csv")
PGD_NOV_RUNS_CSV              = os.path.join(PGD_NOV_RESULTS_DIR, "nov_pgd_morphence_runs.csv")

# Load Novel Students
print("Loading Novel Adv Students...")
fgsm_nov_students = load_adv_students(fgsm_nov_adv_paths)
bim_nov_students  = load_adv_students(bim_nov_adv_paths)
pgd_nov_students  = load_adv_students(pgd_nov_adv_paths)

fgsm_nov_student_names = [os.path.basename(p).replace(".pth", "") for p in fgsm_nov_adv_paths]
bim_nov_student_names  = [os.path.basename(p).replace(".pth", "") for p in bim_nov_adv_paths]
pgd_nov_student_names  = [os.path.basename(p).replace(".pth", "") for p in pgd_nov_adv_paths]

print("Loaded Novel Adv Students:")
print(f"  FGSM: {len(fgsm_nov_students)}")
print(f"  BIM:  {len(bim_nov_students)}")
print(f"  PGD:  {len(pgd_nov_students)}")

Loading Novel Adv Students...


/tmp/ipykernel_399/2283914661.py:61: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


Loaded Novel Adv Students:
  FGSM: 4
  BIM:  4
  PGD:  4


In [ ]:
print("\nFGSM-Nov diversity + transferability + 30 run eval")

df_fgsm_nov_weight_div = pd.DataFrame()
df_fgsm_nov_pred_div = pd.DataFrame()
df_fgsm_nov_xfer_matrix = pd.DataFrame()
df_fgsm_nov_runs = pd.DataFrame()
fgsm_nov_stats = pd.DataFrame()

def _run_fgsm_nov_eval():
    global df_fgsm_nov_weight_div, df_fgsm_nov_pred_div, df_fgsm_nov_xfer_matrix, df_fgsm_nov_runs, fgsm_nov_stats

    if RUN_FGSM_EVAL:
        print("Running FGSM-Nov evaluation (or loading if files exist)...")

        # Weight Diversity
        if os.path.exists(FGSM_NOV_WEIGHT_DIVERSITY_CSV):
            df_fgsm_nov_weight_div = pd.read_csv(FGSM_NOV_WEIGHT_DIVERSITY_CSV)
            print(f"Loaded FGSM-Nov weight diversity from: {FGSM_NOV_WEIGHT_DIVERSITY_CSV}")
        else:
            df_fgsm_nov_weight_div = compute_weight_diversity(fgsm_nov_students, fgsm_nov_student_names, FGSM_NOV_WEIGHT_DIVERSITY_CSV)

        # Prediction Diversity
        if os.path.exists(FGSM_NOV_PRED_DIVERSITY_CSV):
            df_fgsm_nov_pred_div = pd.read_csv(FGSM_NOV_PRED_DIVERSITY_CSV)
            print(f"Loaded FGSM-Nov prediction diversity from: {FGSM_NOV_PRED_DIVERSITY_CSV}")
        else:
            df_fgsm_nov_pred_div = compute_prediction_diversity(fgsm_nov_students, test_dl, FGSM_NOV_PRED_DIVERSITY_CSV)

        # Transfer Matrix
        if os.path.exists(FGSM_NOV_XFER_MATRIX_CSV):
            df_fgsm_nov_xfer_matrix = pd.read_csv(FGSM_NOV_XFER_MATRIX_CSV)
            print(f"Loaded FGSM-Nov transfer matrix from: {FGSM_NOV_XFER_MATRIX_CSV}")
        else:
            df_fgsm_nov_xfer_matrix = compute_transfer_matrix(
                students=fgsm_nov_students,
                names=fgsm_nov_student_names,
                loader=test_dl,
                attack_fn=rfgsm_attack,
                eps_list=FGSM_TRAIN_EPS_LIST,
                extra_kwargs_func=fgsm_kwargs,
                out_csv=FGSM_NOV_XFER_MATRIX_CSV,
                attack_label="nov_rfgsm",
            )

        # 30-run Evaluation
        if os.path.exists(FGSM_NOV_RUNS_CSV) and os.path.exists(FGSM_NOV_STATS_CSV):
            df_fgsm_nov_runs = pd.read_csv(FGSM_NOV_RUNS_CSV)
            fgsm_nov_stats = pd.read_csv(FGSM_NOV_STATS_CSV)
            print(f"Loaded FGSM-Nov 30-run results from: {FGSM_NOV_RUNS_CSV} and {FGSM_NOV_STATS_CSV}")
        else:
            df_fgsm_nov_runs, fgsm_nov_stats = run_morphence_eval(
                attack_name="nov_rfgsm",
                attack_fn=rfgsm_attack,
                eps_list=FGSM_TRAIN_EPS_LIST,
                students=fgsm_nov_students,
                results_csv=FGSM_NOV_RUNS_CSV,
                stats_csv=FGSM_NOV_STATS_CSV,
                extra_kwargs_func=fgsm_kwargs,
                n_runs=30,
            )
    else:
        print("Skipping FGSM-Nov evaluation. Loading results from disk.")
        df_fgsm_nov_weight_div = pd.read_csv(FGSM_NOV_WEIGHT_DIVERSITY_CSV) if os.path.exists(FGSM_NOV_WEIGHT_DIVERSITY_CSV) else pd.DataFrame()
        df_fgsm_nov_pred_div = pd.read_csv(FGSM_NOV_PRED_DIVERSITY_CSV) if os.path.exists(FGSM_NOV_PRED_DIVERSITY_CSV) else pd.DataFrame()
        df_fgsm_nov_xfer_matrix = pd.read_csv(FGSM_NOV_XFER_MATRIX_CSV) if os.path.exists(FGSM_NOV_XFER_MATRIX_CSV) else pd.DataFrame()
        df_fgsm_nov_runs = pd.read_csv(FGSM_NOV_RUNS_CSV) if os.path.exists(FGSM_NOV_RUNS_CSV) else pd.DataFrame()
        fgsm_nov_stats = pd.read_csv(FGSM_NOV_STATS_CSV) if os.path.exists(FGSM_NOV_STATS_CSV) else pd.DataFrame()

run_stage_with_monitor("fgsm_nov_eval", _run_fgsm_nov_eval)



FGSM-Nov diversity + transferability + 30 run eval
Running FGSM-Nov evaluation (or loading if files exist)...
Loaded FGSM-Nov weight diversity from: /content/drive/MyDrive/PES/project/appEnergyNov/fgsm/results/nov_fgsm_weight_diversity.csv
Loaded FGSM-Nov prediction diversity from: /content/drive/MyDrive/PES/project/appEnergyNov/fgsm/results/nov_fgsm_prediction_diversity.csv
Loaded FGSM-Nov transfer matrix from: /content/drive/MyDrive/PES/project/appEnergyNov/fgsm/results/nov_fgsm_transfer_matrix.csv
Loaded FGSM-Nov 30-run results from: /content/drive/MyDrive/PES/project/appEnergyNov/fgsm/results/nov_fgsm_morphence_runs.csv and /content/drive/MyDrive/PES/project/appEnergyNov/fgsm/results/nov_fgsm_morphence_stats.csv


In [ ]:
print("\nBIM-Nov diversity + transferability + 30 run eval")

df_bim_nov_weight_div = pd.DataFrame()
df_bim_nov_pred_div = pd.DataFrame()
df_bim_nov_xfer_matrix = pd.DataFrame()
df_bim_nov_runs = pd.DataFrame()
bim_nov_stats = pd.DataFrame()

def _run_bim_nov_eval():
    global df_bim_nov_weight_div, df_bim_nov_pred_div, df_bim_nov_xfer_matrix, df_bim_nov_runs, bim_nov_stats

    if RUN_BIM_EVAL:
        print("Running BIM-Nov evaluation (or loading if files exist)...")

        # Weight Diversity
        if os.path.exists(BIM_NOV_WEIGHT_DIVERSITY_CSV):
            df_bim_nov_weight_div = pd.read_csv(BIM_NOV_WEIGHT_DIVERSITY_CSV)
            print(f"Loaded BIM-Nov weight diversity from: {BIM_NOV_WEIGHT_DIVERSITY_CSV}")
        else:
            df_bim_nov_weight_div = compute_weight_diversity(bim_nov_students, bim_nov_student_names, BIM_NOV_WEIGHT_DIVERSITY_CSV)

        # Prediction Diversity
        if os.path.exists(BIM_NOV_PRED_DIVERSITY_CSV):
            df_bim_nov_pred_div = pd.read_csv(BIM_NOV_PRED_DIVERSITY_CSV)
            print(f"Loaded BIM-Nov prediction diversity from: {BIM_NOV_PRED_DIVERSITY_CSV}")
        else:
            df_bim_nov_pred_div = compute_prediction_diversity(bim_nov_students, test_dl, BIM_NOV_PRED_DIVERSITY_CSV)

        # Transfer Matrix
        if os.path.exists(BIM_NOV_XFER_MATRIX_CSV):
            df_bim_nov_xfer_matrix = pd.read_csv(BIM_NOV_XFER_MATRIX_CSV)
            print(f"Loaded BIM-Nov transfer matrix from: {BIM_NOV_XFER_MATRIX_CSV}")
        else:
            df_bim_nov_xfer_matrix = compute_transfer_matrix(
                students=bim_nov_students,
                names=bim_nov_student_names,
                loader=test_dl,
                attack_fn=bim_attack,
                eps_list=BIM_TRAIN_EPS_LIST,
                extra_kwargs_func=bim_kwargs,
                out_csv=BIM_NOV_XFER_MATRIX_CSV,
                attack_label="nov_bim",
            )

        # 30-run Evaluation
        if os.path.exists(BIM_NOV_RUNS_CSV) and os.path.exists(BIM_NOV_STATS_CSV):
            df_bim_nov_runs = pd.read_csv(BIM_NOV_RUNS_CSV)
            bim_nov_stats = pd.read_csv(BIM_NOV_STATS_CSV)
            print(f"Loaded BIM-Nov 30-run results from: {BIM_NOV_RUNS_CSV} and {BIM_NOV_STATS_CSV}")
        else:
            df_bim_nov_runs, bim_nov_stats = run_morphence_eval(
                attack_name="nov_bim",
                attack_fn=bim_attack,
                eps_list=BIM_TRAIN_EPS_LIST,
                students=bim_nov_students,
                results_csv=BIM_NOV_RUNS_CSV,
                stats_csv=BIM_NOV_STATS_CSV,
                extra_kwargs_func=bim_kwargs,
                n_runs=30,
            )
    else:
        print("Skipping BIM-Nov evaluation. Loading results from disk.")
        df_bim_nov_weight_div = pd.read_csv(BIM_NOV_WEIGHT_DIVERSITY_CSV) if os.path.exists(BIM_NOV_WEIGHT_DIVERSITY_CSV) else pd.DataFrame()
        df_bim_nov_pred_div = pd.read_csv(BIM_NOV_PRED_DIVERSITY_CSV) if os.path.exists(BIM_NOV_PRED_DIVERSITY_CSV) else pd.DataFrame()
        df_bim_nov_xfer_matrix = pd.read_csv(BIM_NOV_XFER_MATRIX_CSV) if os.path.exists(BIM_NOV_XFER_MATRIX_CSV) else pd.DataFrame()
        df_bim_nov_runs = pd.read_csv(BIM_NOV_RUNS_CSV) if os.path.exists(BIM_NOV_RUNS_CSV) else pd.DataFrame()
        bim_nov_stats = pd.read_csv(BIM_NOV_STATS_CSV) if os.path.exists(BIM_NOV_STATS_CSV) else pd.DataFrame()

run_stage_with_monitor("bim_nov_eval", _run_bim_nov_eval)



BIM-Nov diversity + transferability + 30 run eval
Running BIM-Nov evaluation (or loading if files exist)...
Loaded BIM-Nov weight diversity from: /content/drive/MyDrive/PES/project/appEnergyNov/bim/results/nov_bim_weight_diversity.csv
Loaded BIM-Nov prediction diversity from: /content/drive/MyDrive/PES/project/appEnergyNov/bim/results/nov_bim_prediction_diversity.csv
Loaded BIM-Nov transfer matrix from: /content/drive/MyDrive/PES/project/appEnergyNov/bim/results/nov_bim_transfer_matrix.csv
Loaded BIM-Nov 30-run results from: /content/drive/MyDrive/PES/project/appEnergyNov/bim/results/nov_bim_morphence_runs.csv and /content/drive/MyDrive/PES/project/appEnergyNov/bim/results/nov_bim_morphence_stats.csv


In [ ]:
print("\nPGD-Nov diversity + transferability + 30 run eval")

df_pgd_nov_weight_div = pd.DataFrame()
df_pgd_nov_pred_div = pd.DataFrame()
df_pgd_nov_xfer_matrix = pd.DataFrame()
df_pgd_nov_runs = pd.DataFrame()
pgd_nov_stats = pd.DataFrame()

def _run_pgd_nov_eval():
    global df_pgd_nov_weight_div, df_pgd_nov_pred_div, df_pgd_nov_xfer_matrix, df_pgd_nov_runs, pgd_nov_stats

    if RUN_PGD_EVAL:
        print("Running PGD-Nov evaluation (or loading if files exist)...")

        # Weight Diversity
        if os.path.exists(PGD_NOV_WEIGHT_DIVERSITY_CSV):
            df_pgd_nov_weight_div = pd.read_csv(PGD_NOV_WEIGHT_DIVERSITY_CSV)
            print(f"Loaded PGD-Nov weight diversity from: {PGD_NOV_WEIGHT_DIVERSITY_CSV}")
        else:
            df_pgd_nov_weight_div = compute_weight_diversity(pgd_nov_students, pgd_nov_student_names, PGD_NOV_WEIGHT_DIVERSITY_CSV)

        # Prediction Diversity
        if os.path.exists(PGD_NOV_PRED_DIVERSITY_CSV):
            df_pgd_nov_pred_div = pd.read_csv(PGD_NOV_PRED_DIVERSITY_CSV)
            print(f"Loaded PGD-Nov prediction diversity from: {PGD_NOV_PRED_DIVERSITY_CSV}")
        else:
            df_pgd_nov_pred_div = compute_prediction_diversity(pgd_nov_students, test_dl, PGD_NOV_PRED_DIVERSITY_CSV)

        # Transfer Matrix
        if os.path.exists(PGD_NOV_XFER_MATRIX_CSV):
            df_pgd_nov_xfer_matrix = pd.read_csv(PGD_NOV_XFER_MATRIX_CSV)
            print(f"Loaded PGD-Nov transfer matrix from: {PGD_NOV_XFER_MATRIX_CSV}")
        else:
            df_pgd_nov_xfer_matrix = compute_transfer_matrix(
                students=pgd_nov_students,
                names=pgd_nov_student_names,
                loader=test_dl,
                attack_fn=pgd_attack,
                eps_list=PGD_TRAIN_EPS_LIST,
                extra_kwargs_func=pgd_kwargs,
                out_csv=PGD_NOV_XFER_MATRIX_CSV,
                attack_label="nov_pgd",
            )

        # 30-run Evaluation
        if os.path.exists(PGD_NOV_RUNS_CSV) and os.path.exists(PGD_NOV_STATS_CSV):
            df_pgd_nov_runs = pd.read_csv(PGD_NOV_RUNS_CSV)
            pgd_nov_stats = pd.read_csv(PGD_NOV_STATS_CSV)
            print(f"Loaded PGD-Nov 30-run results from: {PGD_NOV_RUNS_CSV} and {PGD_NOV_STATS_CSV}")
        else:
            df_pgd_nov_runs, pgd_nov_stats = run_morphence_eval(
                attack_name="nov_pgd",
                attack_fn=pgd_attack,
                eps_list=PGD_TRAIN_EPS_LIST,
                students=pgd_nov_students,
                results_csv=PGD_NOV_RUNS_CSV,
                stats_csv=PGD_NOV_STATS_CSV,
                extra_kwargs_func=pgd_kwargs,
                n_runs=30,
            )
    else:
        print("Skipping PGD-Nov evaluation. Loading results from disk.")
        df_pgd_nov_weight_div = pd.read_csv(PGD_NOV_WEIGHT_DIVERSITY_CSV) if os.path.exists(PGD_NOV_WEIGHT_DIVERSITY_CSV) else pd.DataFrame()
        df_pgd_nov_pred_div = pd.read_csv(PGD_NOV_PRED_DIVERSITY_CSV) if os.path.exists(PGD_NOV_PRED_DIVERSITY_CSV) else pd.DataFrame()
        df_pgd_nov_xfer_matrix = pd.read_csv(PGD_NOV_XFER_MATRIX_CSV) if os.path.exists(PGD_NOV_XFER_MATRIX_CSV) else pd.DataFrame()
        df_pgd_nov_runs = pd.read_csv(PGD_NOV_RUNS_CSV) if os.path.exists(PGD_NOV_RUNS_CSV) else pd.DataFrame()
        pgd_nov_stats = pd.read_csv(PGD_NOV_STATS_CSV) if os.path.exists(PGD_NOV_STATS_CSV) else pd.DataFrame()

run_stage_with_monitor("pgd_nov_eval", _run_pgd_nov_eval)



PGD-Nov diversity + transferability + 30 run eval
Running PGD-Nov evaluation (or loading if files exist)...
Saved weight diversity to: /content/drive/MyDrive/PES/project/appEnergyNov/pgd/results/nov_pgd_weight_diversity.csv
Saved prediction diversity to: /content/drive/MyDrive/PES/project/appEnergyNov/pgd/results/nov_pgd_prediction_diversity.csv

[NOV_PGD] Transfer matrix for eps=0.1...
  atk=student_05_enc_attn_pgd_adv - def=student_05_enc_attn_pgd_adv | eps=0.10 | RMSE=0.07899
  atk=student_05_enc_attn_pgd_adv - def=student_06_ffn_pgd_adv | eps=0.10 | RMSE=0.07812
  atk=student_05_enc_attn_pgd_adv - def=student_07_norms_pgd_adv | eps=0.10 | RMSE=0.07801
  atk=student_05_enc_attn_pgd_adv - def=student_08_inproj_head_pgd_adv | eps=0.10 | RMSE=0.07819
  atk=student_06_ffn_pgd_adv - def=student_05_enc_attn_pgd_adv | eps=0.10 | RMSE=0.07875
  atk=student_06_ffn_pgd_adv - def=student_06_ffn_pgd_adv | eps=0.10 | RMSE=0.07840
  atk=student_06_ffn_pgd_adv - def=student_07_norms_pgd_adv | eps

In [ ]:
# config for NOVEL attacks
ATTACK_CONFIGS_NOV = [
    dict(
        name="FGSM-Nov",
        prefix="nov_fgsm",
        attack_key="nov_rfgsm",   # matches attack_label in nov eval
        runs_csv=FGSM_NOV_RUNS_CSV,
        results_dir=FGSM_NOV_RESULTS_DIR,
    ),
    dict(
        name="BIM-Nov",
        prefix="nov_bim",
        attack_key="nov_bim",     # matches attack_label in nov eval
        runs_csv=BIM_NOV_RUNS_CSV,
        results_dir=BIM_NOV_RESULTS_DIR,
    ),
    dict(
        name="PGD-Nov",
        prefix="nov_pgd",
        attack_key="nov_pgd",     # matches attack_label in nov eval
        runs_csv=PGD_NOV_RUNS_CSV,
        results_dir=PGD_NOV_RESULTS_DIR,
    ),
]

# run for FGSM-Nov, BIM-Nov, PGD-Nov
for cfg in ATTACK_CONFIGS_NOV:
    compute_stats_for_attack(cfg)


Processing FGSM-Nov runs
Runs CSV: /content/drive/MyDrive/PES/project/appEnergyNov/fgsm/results/nov_fgsm_morphence_runs.csv

[FGSM-Nov | Clean] stats:
         model     mean  std      min       q1   median       q3      max  n
          base 0.079786  0.0 0.079786 0.079786 0.079786 0.079786 0.079786 30
morph_ensemble 0.075020  0.0 0.075020 0.075020 0.075020 0.075020 0.075020 30
Saved clean stats to: /content/drive/MyDrive/PES/project/appEnergyNov/fgsm/results/nov_fgsm_clean_stats.csv

[FGSM-Nov | nov_rfgsm] base stats:
 epsilon     mean      std      min       q1   median       q3      max  n
     0.1 0.103996 0.000029 0.103937 0.103979 0.103992 0.104011 0.104064 30
     0.2 0.104302 0.000044 0.104234 0.104275 0.104291 0.104329 0.104421 30
     0.3 0.104899 0.000042 0.104811 0.104875 0.104897 0.104922 0.105004 30
     0.5 0.106734 0.000086 0.106543 0.106681 0.106751 0.106793 0.106914 30
Saved: /content/drive/MyDrive/PES/project/appEnergyNov/fgsm/results/nov_fgsm_nov_rfgsm_base_stats.